In [45]:
# ============================================================
# 030_daily_paper_scanner
# ============================================================
#
# Overview
# ----------------
# Daily scanner that fetches recent scholarly papers within a configurable scan window
# (e.g., last 24–48 hours), filters by monitoring themes, performs multi-stage deduplication
# (DOI > arXiv ID > title similarity), generates short summaries, and creates new records
# in the Notion Papers DB with status="INBOX". Designed to be idempotent and safe to rerun.
#
# Inputs / Outputs
# ----------------
# Inputs:
#   - scanner_config (in this notebook):
#       - scan_window_hours, title_similarity_threshold, dry_run
#       - max_papers_per_source
#       - theme keywords / monitoring themes (used by paper_is_relevant + S2 query builder)
#       - arxiv_categories, semantic_scholar_fields
#       - S2 throttling params (s2_page_limit, s2_max_retries, etc.)
#   - 028_config_and_state:
#       - env loading, logger, run_id
#       - state helpers: get_state(), update_state()
#       - last_scan_run for incremental scanning
#   - 029_notion_clients_and_io:
#       - Notion duplicate lookups (find_duplicate_by_*)
#       - Notion create: create_paper_inbox() (preferred) or create_paper()
#   - External APIs:
#       - arXiv API (Atom feed)
#       - Semantic Scholar Graph API (/graph/v1/paper/search)
#
# Outputs:
#   - Notion: new paper records (status="INBOX") with metadata and run_id/dedup_key
#   - State:
#       - last_scan_run updated to scan_window end_time (UTC ISO)
#       - scan statistics persisted (scan_stats_{run_id})
#   - Logs:
#       - fetched counts per source, dedup stats, notion dup-check stats, summary stats, write stats
#
# Pipeline Structure
# ----------------
# Cell 01: Configuration and dependency checks (imports, env, logger)
# Cell 02: Scan window calculation (uses last_scan_run + scan_window_hours)
# Cell 03: Text/title normalization + relevance filtering helpers (paper_is_relevant)
# Cell 04: Dedup key generation (SHA256 over normalized DOI/arXiv/title)
# Cell 05: Title similarity matching (difflib.SequenceMatcher)
# Cell 06: Fetch papers from arXiv API (category + submittedDate window)
# Cell 07: Fetch papers from Semantic Scholar API
#          - theme-keyword-driven query (OR query)
#          - rate-limit handling (429 retry/backoff)
#          - final gating via paper_is_relevant
# Cell 08: Merge and deduplicate fetched batch
# Cell 09: Check Notion DB for existing duplicates (DOI > arXiv > title similarity)
# Cell 10: Generate short summaries (LLM if available; fallback to abstract truncation)
# Cell 11: Create new paper records in Notion
#          - prefers create_paper_inbox() (029), falls back to create_paper()
#          - dry_run supported
#          - defensive argument passing (schema/signature drift tolerant)
# Cell 12: Persist state and report final statistics (KeyError-safe dict access)
#
# Notes / Guarantees
# ----------------
# - Dedup hierarchy: DOI > arXiv ID > normalized title (all hashed with SHA256)
# - Title similarity threshold is configurable (default 0.85)
# - Semantic Scholar is rate-limited; API key (SEMANTIC_SCHOLAR_API_KEY) recommended
# - Robustness:
#     - None-safe parsing for API responses
#     - 429 handling with Retry-After + exponential backoff + jitter
#     - config/scanner_config access is defensive (avoids KeyError)
# - Idempotency strategy:
#     - within-run dedup via dedup_key + title similarity
#     - cross-run dedup via Notion duplicate checks
# - Observability:
#     - all new records include run_id and dedup fields
#     - scan summary persisted to state for tracking/monitoring
# - Summary generation:
#     - prefers LLM when configured; otherwise uses abstract truncation fallback


In [33]:
# ============================================================
# Cell 01 — Preflight: load 028/029 context + validate readiness
# ============================================================
# Overview:
#   Ensures notebook dependencies (028/029) are executed,
#   validates required runtime objects exist (run_id/logger/config/state + Notion wrappers),
#   and configures scan parameters for this run.
#
# Inputs / Outputs:
#   Inputs: 028_config_and_state.ipynb, 029_notion_clients_and_io.ipynb
#   Outputs: run-scoped config for scanner; validated availability of required helpers
#
# Notes:
#   - Do NOT load env.txt here (028 owns env/config)
#   - Do NOT initialize Notion client here (029 owns Notion I/O)
#   - Fail fast with actionable messages if 028/029 are not available
#

# --- Execute upstream notebooks (preferred Jupyter pattern) ---
# Adjust paths if these notebooks are not in the same directory.
try:
    get_ipython().run_line_magic("run", "028_config_and_state.ipynb")
except Exception as e:
    raise RuntimeError(
        "Failed to execute 028_config_and_state.ipynb. "
        "Make sure the file exists and runs cleanly before running 030."
    ) from e

try:
    get_ipython().run_line_magic("run", "029_notion_clients_and_io.ipynb")
except Exception as e:
    raise RuntimeError(
        "Failed to execute 029_notion_clients_and_io.ipynb. "
        "Make sure the file exists and runs cleanly before running 030."
    ) from e

logger.info("029 exports snapshot: " + ", ".join(sorted([k for k,v in globals().items() if callable(v) and ("paper" in k or "papers" in k)]))[:500])

# --- Validate that 028 provided core runtime objects ---
required_028_symbols = ["run_id", "logger", "config", "load_state", "save_state"]
missing_028 = [name for name in required_028_symbols if name not in globals()]
if missing_028:
    raise RuntimeError(
        f"028 did not expose required symbols: {missing_028}. "
        "Expected 028 to define: run_id, logger, config, load_state, save_state."
    )

logger.info(f"Starting daily paper scanner run: {run_id}")
# --- Canonicalize 029 exports (map to names 030 expects) ---
if "create_paper" not in globals() and "create_paper_inbox" in globals():
    create_paper = create_paper_inbox
    logger.info("Aliased create_paper <- create_paper_inbox (from 029)")

# --- Validate that 029 provided Notion wrappers we will use ---
required_029_symbols = [
    "create_paper_inbox",
    "create_paper",
    "find_duplicate_paper",        # or your actual helper name(s)
    "validate_database_schema",    # optional but recommended
]
missing_029 = [name for name in required_029_symbols if name not in globals()]
if missing_029:
    # Don’t silently stub. Fail and fix 029 exports instead.
    raise RuntimeError(
        f"029 did not expose required symbols: {missing_029}. "
        "Ensure 029 defines and exposes the Papers DB wrappers (create_paper, find_duplicate helpers)."
    )

# --- Scanner configuration (domain logic only) ---

THEMES: Dict[str, List[str]] = {
    "venture_capital": [
        # core
        "venture capital", "vc", "venture investing", "venture investment",
        "venture fund", "venture financing", "startup financing",
        "early-stage financing", "seed financing", "series a", "series b",
        "growth equity", "scale-up", "scaleup",

        # instruments / mechanics
        "term sheet", "valuation", "cap table", "dilution",
        "convertible note", "safe", "equity financing",
        "syndicate", "lead investor", "co-investment", "co-invest",

        # lifecycle / outcomes
        "exit", "liquidity event", "ipo", "acquisition",
        "m&a", "merger", "buyout", "secondary sale",

        # ecosystem
        "accelerator", "incubator", "angel investor", "angel investing",
        "startup ecosystem", "entrepreneurial ecosystem"
    ],

    "limited_partners": [
        # core LP terms
        "limited partner", "lp", "general partner", "gp",
        "institutional investor", "asset owner",

        # LP types
        "pension fund", "public pension", "sovereign wealth fund", "swf",
        "endowment", "university endowment", "family office",
        "insurance company", "foundation",

        # allocation / diligence / fund ops
        "fund of funds", "commitment", "capital commitment",
        "capital call", "distribution", "dpi", "tvpi", "irr",
        "portfolio allocation", "asset allocation",
        "due diligence", "fund performance", "manager selection"
    ],

    "government_vc": [
        # core
        "government venture capital", "public venture capital",
        "state-backed venture capital", "state venture capital",
        "sovereign wealth fund", "development finance institution", "dfi",
        "development bank",

        # policy vehicles / programs
        "innovation agency", "public investment fund",
        "national innovation", "innovation fund",
        "public-private fund", "public private partnership", "ppp",
        "matching fund", "co-investment program", "co-investment scheme",

        # related policy finance
        "industrial development", "strategic investment",
        "technology transfer office", "tto"
    ],

    "entrepreneurship_policy": [
        # core
        "entrepreneurship policy", "innovation policy", "startup policy",
        "industrial policy", "technology policy", "science policy",

        # regulation / taxation
        "regulation", "regulatory", "regulatory sandbox",
        "tax incentive", "tax credit", "capital gains tax",
        "stock option", "employee stock ownership", "esop",

        # public support tools
        "public subsidy", "grant program", "r&d subsidy", "r&d grant",
        "public procurement", "government procurement",
        "innovation voucher", "cluster policy", "technology cluster",
        "startup visa", "immigration policy", "talent policy",

        # commercialization
        "university spinout", "spin-out", "spin-off", "technology commercialization",
        "knowledge transfer"
    ],
}

# optional: negative keywords to reduce off-theme papers
NEGATIVE_KEYWORDS = [
    "prep", "msm", "vaccine", "opioid", "clinical", "patient",
    "machiavellianism", "personality", "psychology", "dietary guideline",
]

scanner_config = {
    "scan_window_hours": 48,
    "title_similarity_threshold": 0.85,
    "dry_run": False,
    "max_papers_per_source": 100,

    # ★テーマ定義を構造化
    "theme_keywords": THEMES,

    # ★最低1語はヒットしてほしい「コア条件」
    "required_theme_groups": [
        "venture_capital",
        "government_vc",
        "entrepreneurship_policy",
        "limited_partners",
    ],

    # ★ノイズ除去
    "negative_keywords": [
        "prep", "vaccine", "opioid", "clinical",
        "psychology", "personality", "dietary",
    ],

    "arxiv_categories": ["econ.GN", "cs.CY", "physics.soc-ph"],
    "semantic_scholar_fields": ["Economics", "Business", "Political Science", "Sociology"],
}



# --- Optional: validate Notion schema (029 responsibility) ---
try:
    import inspect
    sig = inspect.signature(validate_database_schema)
    if len(sig.parameters) == 0:
        validate_database_schema()
    else:
        validate_database_schema("papers")
    logger.info("Notion schema validated (via 029).")
except Exception as e:
    logger.warning(f"Schema validation warning (via 029): {e}")


logger.info(f"Scanner configuration: {scanner_config}")
logger.info("Cell 01 complete: 028/029 loaded and readiness validated.")


✓ Cell 01: Imports and dependencies loaded
✓ Cell 02: Environment bootstrap completed
  - NOTION_TOKEN: ntn_38...***
  - NOTION_VERSION: 2025-09-03
  - NOTION_LIT_DB_ID: set
  - NOTION_EVENTS_DB_ID: set
  - NOTION_MONITORING_TARGETS_DB_ID: set
  - NOTION_MONITORING_QUEUE_DB_ID: set
✓ Cell 03: Loaded configuration from config.yaml
✓ Cell 03: Configuration validated (source: config.yaml)
  - Pipeline cadence:     daily
  - Max runtime:          30 min
  - Lookback (daily):     7 days
  - Lookback (tasks):     14 days
  - Lookback (projects):  30 days
  - Max items per query:  100
  - Logging level:        INFO
✓ Cell 04: Run context initialized
  - Run ID:           2ac5680b-7a63-488f-a211-1860c73ee2f8
  - Execution start:  2026-01-25T05:10:37.609081+00:00
  - Run date:         2026-01-25
  - Timezone:         UTC
  - Cadence:          daily
  - Lookback windows:
    - daily_notes : 7 days (2026-01-18 to 2026-01-25)
    - tasks       : 14 days (2026-01-11 to 2026-01-25)
    - projects   

In [34]:
# ============================================================
# Cell 02 — Time window calculation
# ============================================================
# Overview:
#   Computes the time window for scanning using scanner_config['scan_window_hours'].
#   Uses the registered state key 'cursor_daily' for incremental scanning.
#
# Inputs / Outputs:
#   Inputs: scanner_config['scan_window_hours'], load_state('cursor_daily')
#   Outputs: start_time, end_time, window_hours
#
# Notes:
#   - This project uses registered state keys only (see state_paths)
#   - We store a dedicated timestamp under cursor_daily["papers_last_scan_run"]
#

from datetime import datetime, timedelta, timezone

end_time = datetime.now(timezone.utc)

# Registered key (allowed by your 028 state system)
cursor_daily = load_state("cursor_daily", default={}) or {}

# We store our scanner cursor inside this dict
last_scan_run = cursor_daily.get("papers_last_scan_run")

scan_hours = int(scanner_config.get("scan_window_hours", 48))

if last_scan_run:
    try:
        last_run_dt = datetime.fromisoformat(str(last_scan_run).replace("Z", "+00:00"))
        max_lookback = timedelta(days=7)

        if end_time - last_run_dt < max_lookback:
            start_time = last_run_dt
            logger.info(f"Using incremental scan from last run: {last_run_dt.isoformat()}")
        else:
            start_time = end_time - timedelta(hours=scan_hours)
            logger.warning(f"Last run too old ({last_run_dt.isoformat()}); using {scan_hours}h window")
    except Exception as e:
        start_time = end_time - timedelta(hours=scan_hours)
        logger.warning(f"Could not parse last_scan_run '{last_scan_run}': {e}; using {scan_hours}h window")
else:
    start_time = end_time - timedelta(hours=scan_hours)
    logger.info(f"No prior run found; using configured scan window: {scan_hours}h")

window_hours = (end_time - start_time).total_seconds() / 3600
logger.info(f"Scan window: {start_time.isoformat()} to {end_time.isoformat()} ({window_hours:.1f} hours)")

scan_window = {"start_time": start_time, "end_time": end_time, "window_hours": window_hours}
logger.info("Cell 02 complete: time window calculated")


2026-01-25 14:10:59 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 | No prior run found; using configured scan window: 48h
2026-01-25 14:10:59 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 | Scan window: 2026-01-23T05:10:59.897760+00:00 to 2026-01-25T05:10:59.897760+00:00 (48.0 hours)
2026-01-25 14:10:59 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 | Cell 02 complete: time window calculated


In [35]:
# ============================================================
# Cell 03 — Title normalization utilities
# ============================================================
# Overview:
#   Provides functions for normalizing paper titles to enable robust similarity matching
#   and deduplication. Handles case normalization, punctuation removal, whitespace
#   collapsing, and common academic title prefixes/suffixes.
#
# Inputs / Outputs:
#   Inputs: raw title string
#   Outputs: normalized title string (lowercase, stripped, collapsed whitespace)
#
# Notes:
#   - Used by title similarity matching (Cell 05) and dedup key generation (Cell 04)
#   - Removes common academic noise: "the", "a", "an" (configurable)
#   - Preserves semantic content while enabling fuzzy matching
#   - Deterministic output for consistent dedup_key generation
#

import unicodedata
def normalize_text(text: str) -> str:
    if not text:
        return ""
    return re.sub(r"\s+", " ", text.lower()).strip()


def paper_is_relevant(text: str, config: dict) -> bool:
    """
    Determine whether a paper is relevant to monitoring themes.
    
    Strategy:
      1. Negative keyword exclusion (hard filter)
      2. Positive theme keyword matching (soft requirement)
    
    Returns:
      True if relevant, False otherwise
    """
    if not text:
        return False

    text_n = normalize_text(text)

    # --- Hard negative filter ---
    for ng in config.get("negative_keywords", []):
        if ng.lower() in text_n:
            return False

    # --- Theme matching ---
    theme_keywords = config.get("theme_keywords", {})
    required_groups = config.get("required_theme_groups", [])

    if not theme_keywords or not required_groups:
        # Fallback: allow all if not configured
        return True

    matched_groups = 0
    for group in required_groups:
        keywords = theme_keywords.get(group, [])
        if any(k.lower() in text_n for k in keywords):
            matched_groups += 1

    return matched_groups >= 1
def normalize_title(title: str, aggressive: bool = False) -> str:
    """
    Normalize a paper title for comparison and deduplication.
    
    Args:
        title: Raw paper title string
        aggressive: If True, applies more aggressive normalization (removes articles, etc.)
    
    Returns:
        Normalized title string
    
    Example:
        >>> normalize_title("The Impact of AI on Society: A Review")
        'impact of ai on society review'
    """
    if not title or not isinstance(title, str):
        return ""
    
    # 1. Unicode normalization (NFKD decomposition)
    normalized = unicodedata.normalize('NFKD', title)
    # After NFKD, drop diacritics (accent marks)
    normalized = "".join(ch for ch in normalized if not unicodedata.combining(ch))
    
    # 2. Convert to lowercase
    normalized = normalized.lower()
    
    # 3. Remove common punctuation (keep spaces, hyphens, alphanumerics)
    # Replace punctuation with spaces to avoid word concatenation
    import string
    punct_to_space = str.maketrans(string.punctuation, ' ' * len(string.punctuation))
    normalized = normalized.translate(punct_to_space)
    
    # 4. Collapse multiple whitespace to single space
    normalized = ' '.join(normalized.split())
    
    # 5. Aggressive mode: remove common articles and stop words
    if aggressive:
        stop_words = {'a', 'an', 'the', 'and', 'or', 'but', 'of', 'in', 'on', 'at', 'to', 'for'}
        words = normalized.split()
        words = [w for w in words if w not in stop_words]
        normalized = ' '.join(words)
    
    # 6. Strip leading/trailing whitespace
    normalized = normalized.strip()
    
    return normalized


def normalize_title_for_dedup(title: str) -> str:
    """
    Specialized normalization for deduplication key generation.
    Uses aggressive normalization for maximum collision on similar titles.
    
    Args:
        title: Raw paper title string
    
    Returns:
        Aggressively normalized title for dedup_key generation
    """
    return normalize_title(title, aggressive=True)


def normalize_title_for_similarity(title: str) -> str:
    """
    Specialized normalization for similarity matching (less aggressive).
    Preserves more structure for difflib comparison.
    
    Args:
        title: Raw paper title string
    
    Returns:
        Normalized title for similarity comparison
    """
    return normalize_title(title, aggressive=False)


# --- Test cases ---
RUN_TITLE_NORMALIZATION_TESTS = True

if RUN_TITLE_NORMALIZATION_TESTS:
    test_titles = [
        "The Impact of AI on Society: A Review",
        "Impact of AI on Society - A Review!",
        "  THE IMPACT OF AI ON SOCIETY: A REVIEW  ",
        "Café Society: An Étude in Résumé",
    ]

    logger.info("Testing title normalization:")
    for title in test_titles:
        normal = normalize_title_for_similarity(title)
        dedup = normalize_title_for_dedup(title)
        logger.info(f"Original: '{title}'")
        logger.info(f"  Similarity: '{normal}'")
        logger.info(f"  Dedup: '{dedup}'")
        logger.info("")


logger.info("Cell 03 complete: title normalization utilities defined")


2026-01-25 14:11:06 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 | Testing title normalization:
2026-01-25 14:11:06 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 | Original: 'The Impact of AI on Society: A Review'
2026-01-25 14:11:06 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 |   Similarity: 'the impact of ai on society a review'
2026-01-25 14:11:06 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 |   Dedup: 'impact ai society review'
2026-01-25 14:11:06 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 | 
2026-01-25 14:11:06 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 | Original: 'Impact of AI on Society - A Review!'
2026-01-25 14:11:06 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 |   Similarity: 'impact of ai on society a review'
2026-01-25 14:11:06 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 |   Dedup: 'impact ai society review'
2026-01-25 14:11:06 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 | 
2026-01-25 14:11:06 | INFO     | 2ac5680b-7a63-48

In [36]:
# ============================================================
# Cell 04 — Deduplication key generation
# ============================================================
# Overview:
#   Generates deterministic deduplication keys for paper records using a hierarchical
#   strategy: DOI (preferred) > arXiv ID > normalized title. Keys are SHA256 hashes
#   to ensure consistent length and avoid collisions.
#
# Inputs / Outputs:
#   Inputs: paper dict with optional 'doi', 'arxiv_id', 'title' fields
#   Outputs: dedup_key (str), dedup_method (str)
#
# Notes:
#   - Dedup key hierarchy ensures stable identity across sources
#   - SHA256 provides cryptographic collision resistance
#   - Normalized title fallback enables fuzzy matching via separate similarity check
#   - All keys are lowercase hex strings (64 chars)
#   - Used by Cell 08 (batch dedup) and Cell 09 (Notion duplicate checks)
#
import hashlib
def generate_dedup_key(paper: Dict[str, Any]) -> tuple[str, str]:
    """
    Generate a deterministic deduplication key for a paper record.
    
    Hierarchy:
      1. DOI (if present and non-empty)
      2. arXiv ID (if present and non-empty)
      3. Normalized title (fallback)
    
    Args:
        paper: Dict with optional 'doi', 'arxiv_id', 'title' keys
    
    Returns:
        Tuple of (dedup_key: str, dedup_method: str)
        - dedup_key: SHA256 hex digest (64 chars)
        - dedup_method: 'doi' | 'arxiv_id' | 'title'
    
    Raises:
        ValueError: If paper has no valid identifier (no DOI, arXiv ID, or title)
    
    Example:
        >>> paper = {'doi': '10.1234/example', 'title': 'Test Paper'}
        >>> key, method = generate_dedup_key(paper)
        >>> method
        'doi'
        >>> len(key)
        64
    """
    # 1. Try DOI first (preferred)
    doi = paper.get('doi', '').strip()
    if doi:
        # Normalize DOI: lowercase, remove URL prefixes
        doi_normalized = doi.lower()
        doi_normalized = doi_normalized.replace('https://doi.org/', '')
        doi_normalized = doi_normalized.replace('http://dx.doi.org/', '')
        doi_normalized = doi_normalized.replace('doi:', '')
        doi_normalized = doi_normalized.strip()
        
        if doi_normalized:
            key = hashlib.sha256(doi_normalized.encode('utf-8')).hexdigest()
            return (key, 'doi')
    
    # 2. Try arXiv ID second
    arxiv_id = paper.get('arxiv_id', '').strip()
    if arxiv_id:
        # Normalize arXiv ID: lowercase, remove version suffix if present
        arxiv_normalized = arxiv_id.lower()
        # Handle both old (e.g., 'hep-th/9901001') and new (e.g., '1234.5678v2') formats
        arxiv_normalized = re.sub(r'v\d+$', '', arxiv_normalized)  # Remove version
        arxiv_normalized = arxiv_normalized.strip()
        
        if arxiv_normalized:
            key = hashlib.sha256(arxiv_normalized.encode('utf-8')).hexdigest()
            return (key, 'arxiv_id')
    
    # 3. Fallback to normalized title
    title = paper.get('title', '').strip()
    if title:
        # Use aggressive normalization for dedup key
        title_normalized = normalize_title_for_dedup(title)
        
        if title_normalized:
            key = hashlib.sha256(title_normalized.encode('utf-8')).hexdigest()
            return (key, 'title')
    
    # No valid identifier found
    raise ValueError(f"Paper has no valid identifier for dedup key generation: {paper.get('title', 'NO_TITLE')[:50]}")


def add_dedup_key_to_paper(paper: Dict[str, Any]) -> Dict[str, Any]:
    """
    Augment a paper dict with dedup_key and dedup_method fields.
    Modifies the paper dict in-place and returns it.
    
    Args:
        paper: Paper dict (will be modified in-place)
    
    Returns:
        The same paper dict with added 'dedup_key' and 'dedup_method' fields
    
    Example:
        >>> paper = {'title': 'Test', 'doi': '10.1234/test'}
        >>> add_dedup_key_to_paper(paper)
        {'title': 'Test', 'doi': '10.1234/test', 'dedup_key': '...', 'dedup_method': 'doi'}
    """
    try:
        dedup_key, dedup_method = generate_dedup_key(paper)
        paper['dedup_key'] = dedup_key
        paper['dedup_method'] = dedup_method
    except ValueError as e:
        logger.warning(f"Could not generate dedup key: {e}")
        paper['dedup_key'] = None
        paper['dedup_method'] = None
    
    return paper


def batch_add_dedup_keys(papers: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """
    Add dedup keys to a batch of papers.
    
    Args:
        papers: List of paper dicts
    
    Returns:
        List of papers with dedup_key and dedup_method fields added
    
    Notes:
        - Logs summary statistics of dedup methods used
        - Skips papers that fail key generation (logs warning)
    """
    method_counts = {'doi': 0, 'arxiv_id': 0, 'title': 0, 'failed': 0}
    
    for paper in papers:
        add_dedup_key_to_paper(paper)
        method = paper.get('dedup_method')
        if method:
            method_counts[method] = method_counts.get(method, 0) + 1
        else:
            method_counts['failed'] += 1
    
    logger.info(f"Dedup key generation complete: {method_counts}")
    
    return papers


# --- Test cases ---
RUN_DEDUP_KEY_TESTS = True

if RUN_DEDUP_KEY_TESTS:
    test_papers = [
        {'doi': '10.1234/example', 'title': 'Test Paper with DOI'},
        {'arxiv_id': '2301.12345v2', 'title': 'Test Paper with arXiv ID'},
        {'title': 'Test Paper with Only Title'},
        {'doi': 'https://doi.org/10.5678/test', 'arxiv_id': '2302.99999', 'title': 'Paper with Multiple IDs'},
    ]
    
    logger.info("Testing dedup key generation:")
    for paper in test_papers:
        try:
            key, method = generate_dedup_key(paper)
            logger.info(f"Paper: {paper.get('title', 'NO_TITLE')[:40]}")
            logger.info(f"  Method: {method}")
            logger.info(f"  Key: {key[:16]}...{key[-8:]}")
        except ValueError as e:
            logger.warning(f"Failed: {e}")
        logger.info("")
    
    # Test batch processing
    logger.info("Testing batch dedup key addition:")
    batch_add_dedup_keys(test_papers)
    for paper in test_papers:
        logger.info(f"{paper.get('title', 'NO_TITLE')[:40]}: {paper.get('dedup_method', 'N/A')}")

logger.info("Cell 04 complete: deduplication key generation utilities defined")


2026-01-25 14:11:13 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 | Testing dedup key generation:
2026-01-25 14:11:13 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 | Paper: Test Paper with DOI
2026-01-25 14:11:13 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 |   Method: doi
2026-01-25 14:11:13 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 |   Key: 68b8f7c42b3c20b5...d3f70d78
2026-01-25 14:11:13 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 | 
2026-01-25 14:11:13 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 | Paper: Test Paper with arXiv ID
2026-01-25 14:11:13 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 |   Method: arxiv_id
2026-01-25 14:11:13 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 |   Key: 8d0de5f77dac6d02...734066b2
2026-01-25 14:11:13 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 | 
2026-01-25 14:11:13 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 | Paper: Test Paper with Only Title
2026-01-25 14:11:13 | INFO     | 2ac5680b-7a63-488f-

In [37]:
# ============================================================
# Cell 05 — Title similarity matching
# ============================================================
# Overview:
#   Implements title-based similarity matching using difflib.SequenceMatcher
#   to identify near-duplicate papers with slightly different titles.
#   Used as a fallback when DOI/arXiv ID matching fails.
#
# Inputs / Outputs:
#   Inputs: two title strings, similarity threshold (float, default 0.85)
#   Outputs: similarity score (float 0.0-1.0), boolean match result
#
# Notes:
#   - Uses normalize_title_for_similarity (from Cell 03) for preprocessing
#   - SequenceMatcher ratio provides character-level similarity (0.0-1.0)
#   - Default threshold 0.85 balances precision/recall for academic titles
#   - Case-insensitive, punctuation-normalized comparison
#   - Used by Cell 09 (Notion duplicate checks) and Cell 08 (batch dedup)
#

from difflib import SequenceMatcher
from typing import Any, Dict, List, Optional

def compute_title_similarity(title1: str, title2: str) -> float:
    """
    Compute character-level similarity between two paper titles.
    
    Args:
        title1: First paper title (raw)
        title2: Second paper title (raw)
    
    Returns:
        Similarity score between 0.0 (no match) and 1.0 (exact match)
    
    Example:
        >>> compute_title_similarity("The Impact of AI", "Impact of AI: A Study")
        0.87
    """
    # Normalize both titles for fair comparison
    norm1 = normalize_title_for_similarity(title1)
    norm2 = normalize_title_for_similarity(title2)
    
    # Handle empty titles
    if not norm1 or not norm2:
        return 0.0
    
    # Compute similarity using SequenceMatcher
    matcher = SequenceMatcher(None, norm1, norm2)
    similarity = matcher.ratio()
    
    return similarity


def titles_are_similar(
    title1: str,
    title2: str,
    threshold: Optional[float] = None
) -> tuple[bool, float]:
    """
    Determine if two titles are similar enough to be considered duplicates.
    
    Args:
        title1: First paper title (raw)
        title2: Second paper title (raw)
        threshold: Similarity threshold (0.0-1.0); uses config default if None
    
    Returns:
        Tuple of (is_similar: bool, similarity_score: float)
    
    Example:
        >>> is_similar, score = titles_are_similar("AI and Society", "AI & Society")
        >>> is_similar
        True
        >>> score > 0.85
        True
    """
    # Use configured threshold if not provided
    if threshold is None:
        threshold = scanner_config['title_similarity_threshold']
    
    # Compute similarity
    similarity = compute_title_similarity(title1, title2)
    
    # Check against threshold
    is_similar = similarity >= threshold
    
    return (is_similar, similarity)


def find_similar_title_in_batch(
    target_title: str,
    papers: List[Dict[str, Any]],
    threshold: Optional[float] = None
) -> Optional[tuple[Dict[str, Any], float]]:
    """
    Search a batch of papers for a title similar to the target.
    Returns the first match above threshold.
    
    Args:
        target_title: Title to search for
        papers: List of paper dicts (each must have 'title' key)
        threshold: Similarity threshold; uses config default if None
    
    Returns:
        Tuple of (matching_paper: dict, similarity_score: float) if found,
        None if no match above threshold
    
    Notes:
        - Returns first match only (not highest scoring)
        - Papers without 'title' field are skipped
        - Logs warning if multiple high-similarity matches found
    """
    if threshold is None:
        threshold = scanner_config['title_similarity_threshold']
    
    # Normalize target title once
    target_normalized = normalize_title_for_similarity(target_title)
    
    if not target_normalized:
        return None
    
    matches = []
    
    for paper in papers:
        paper_title = paper.get('title', '').strip()
        if not paper_title:
            continue
        
        # Compute similarity
        similarity = compute_title_similarity(target_title, paper_title)
        
        if similarity >= threshold:
            matches.append((paper, similarity))
    
    if not matches:
        return None
    
    # Sort by similarity (descending) and return best match
    matches.sort(key=lambda x: x[1], reverse=True)
    best_match = matches[0]
    
    # Log if multiple high-quality matches found (potential issue)
    if len(matches) > 1 and matches[1][1] >= threshold + 0.05:
        logger.warning(
            f"Multiple similar titles found for '{target_title[:50]}...': "
            f"{len(matches)} matches (best: {best_match[1]:.3f}, second: {matches[1][1]:.3f})"
        )
    
    return best_match


def batch_find_similar_titles(
    papers: List[Dict[str, Any]],
    threshold: Optional[float] = None
) -> Dict[str, List[tuple[int, int, float]]]:
    """
    Find all similar title pairs within a batch of papers.
    Used for within-batch deduplication.
    
    Args:
        papers: List of paper dicts with 'title' field
        threshold: Similarity threshold; uses config default if None
    
    Returns:
        Dict mapping 'duplicates' to list of (index_i, index_j, similarity_score)
        tuples representing similar pairs
    
    Notes:
        - Performs O(n²) pairwise comparison; efficient for batches < 1000 papers
        - Only reports each pair once (i < j)
        - Used by Cell 08 for batch deduplication
    """
    if threshold is None:
        threshold = scanner_config['title_similarity_threshold']
    
    duplicates = []
    n = len(papers)
    
    # Pairwise comparison
    for i in range(n):
        title_i = papers[i].get('title', '').strip()
        if not title_i:
            continue
        
        for j in range(i + 1, n):
            title_j = papers[j].get('title', '').strip()
            if not title_j:
                continue
            
            # Compute similarity
            similarity = compute_title_similarity(title_i, title_j)
            
            if similarity >= threshold:
                duplicates.append((i, j, similarity))
    
    logger.info(f"Found {len(duplicates)} similar title pairs in batch of {n} papers")
    
    return {'duplicates': duplicates}


# --- Test cases ---
if __name__ == "__main__":
    test_cases = [
        (
            "The Impact of Artificial Intelligence on Society",
            "Impact of Artificial Intelligence on Society: A Review",
            True
        ),
        (
            "Machine Learning for Drug Discovery",
            "Deep Learning for Drug Discovery",
            False  # Different enough to not match
        ),
        (
            "COVID-19 Vaccine Efficacy",
            "COVID19 Vaccine Efficacy",
            True  # Punctuation difference
        ),
        (
            "Quantum Computing",
            "Classical Computing",
            False
        ),
    ]
    
    logger.info("Testing title similarity matching:")
    for title1, title2, expected_similar in test_cases:
        is_similar, score = titles_are_similar(title1, title2)
        status = "✓" if is_similar == expected_similar else "✗"
        logger.info(f"{status} '{title1[:30]}...' vs '{title2[:30]}...'")
        logger.info(f"  Score: {score:.3f}, Similar: {is_similar}, Expected: {expected_similar}")
        logger.info("")
    
    # Test batch finding
    test_batch = [
        {'title': 'The Impact of AI on Society'},
        {'title': 'Impact of AI on Society'},  # Duplicate of [0]
        {'title': 'Machine Learning Methods'},
        {'title': 'Deep Learning Techniques'},  # Not similar to [2]
        {'title': 'The Impact of AI on Society: A Study'},  # Duplicate of [0] and [1]
    ]
    
    logger.info("Testing batch duplicate detection:")
    results = batch_find_similar_titles(test_batch)
    for i, j, score in results['duplicates']:
        logger.info(f"Pair ({i}, {j}): score={score:.3f}")
        logger.info(f"  [{i}]: {test_batch[i]['title']}")
        logger.info(f"  [{j}]: {test_batch[j]['title']}")
        logger.info("")

logger.info("Cell 05 complete: title similarity matching utilities defined")


2026-01-25 14:11:14 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 | Testing title similarity matching:
2026-01-25 14:11:14 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 | ✓ 'The Impact of Artificial Intel...' vs 'Impact of Artificial Intellige...'
2026-01-25 14:11:14 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 |   Score: 0.871, Similar: True, Expected: True
2026-01-25 14:11:14 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 | 
2026-01-25 14:11:14 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 | ✗ 'Machine Learning for Drug Disc...' vs 'Deep Learning for Drug Discove...'
2026-01-25 14:11:14 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 |   Score: 0.866, Similar: True, Expected: False
2026-01-25 14:11:14 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 | 
2026-01-25 14:11:14 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 | ✓ 'COVID-19 Vaccine Efficacy...' vs 'COVID19 Vaccine Efficacy...'
2026-01-25 14:11:14 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 |   

In [38]:
# ============================================================
# Cell 06 — Fetch papers from arXiv API (FULL REPLACEMENT)
# ============================================================
# Adds:
#   - Relevance filtering via paper_is_relevant() using scanner_config
#   - Pagination (start + max_results) to actually reach max_papers
#   - More defensive None-safe parsing
#   - Optional filtering by submitted/published time window (best effort)
#
# Requires (defined in an earlier cell, recommended Cell 03):
#   - paper_is_relevant(text: str, config: dict) -> bool
#   - normalize_text / any helpers used by paper_is_relevant (if applicable)
#
# Outputs:
#   arxiv_papers: List[Dict[str, Any]]
# ============================================================

import xml.etree.ElementTree as ET
from urllib.parse import urlencode

def fetch_arxiv_papers(
    start_time: datetime,
    end_time: datetime,
    categories: List[str],
    max_papers: int = 100
) -> List[Dict[str, Any]]:
    """
    Fetch recent papers from arXiv API within the specified time window and categories.
    Applies relevance filtering (paper_is_relevant) before appending.

    Notes:
      - arXiv query supports submittedDate range but can be strict; we use it as primary filter.
      - We additionally best-effort filter by 'published' parsed datetime if present.
      - Pagination: repeats requests until max_papers per category or no more results.
      - Rate limit: 3 seconds between requests.
    """
    base_url = "http://export.arxiv.org/api/query"

    ns = {
        "atom": "http://www.w3.org/2005/Atom",
        "arxiv": "http://arxiv.org/schemas/atom",
    }

    def _s(x):
        if x is None:
            return ""
        if not isinstance(x, str):
            x = str(x)
        return x.strip()

    all_papers: List[Dict[str, Any]] = []

    # arXiv submittedDate query expects YYYYMMDDHHMMSS
    start_str = start_time.strftime("%Y%m%d%H%M%S")
    end_str   = end_time.strftime("%Y%m%d%H%M%S")

    # arXiv recommends <=100 per request
    page_size = min(max_papers, 100)

    for category in categories:
        logger.info(f"Fetching arXiv papers for category: {category}")

        query = f"cat:{category} AND submittedDate:[{start_str} TO {end_str}]"

        fetched_for_cat = 0
        start_offset = 0

        while fetched_for_cat < max_papers:
            params = {
                "search_query": query,
                "start": start_offset,
                "max_results": page_size,
                "sortBy": "submittedDate",
                "sortOrder": "descending",
            }

            url = f"{base_url}?{urlencode(params)}"
            logger.info(f"arXiv API request: cat={category}, start={start_offset}, max={page_size}")

            try:
                response = requests.get(url, timeout=30)
                response.raise_for_status()
                root = ET.fromstring(response.content)

                entries = root.findall("atom:entry", ns)
                if not entries:
                    logger.info(f"No more results for category {category} (start={start_offset})")
                    break

                logger.info(f"Retrieved {len(entries)} entries for category {category}")

                added_this_page = 0

                for entry in entries:
                    try:
                        paper: Dict[str, Any] = {}

                        title_elem = entry.find("atom:title", ns)
                        title = _s(title_elem.text).replace("\n", " ") if title_elem is not None else ""

                        summary_elem = entry.find("atom:summary", ns)
                        abstract = _s(summary_elem.text).replace("\n", " ") if summary_elem is not None else ""

                        id_elem = entry.find("atom:id", ns)
                        if id_elem is not None:
                            arxiv_url = _s(id_elem.text)
                            arxiv_id = arxiv_url.split("/abs/")[-1] if "/abs/" in arxiv_url else ""
                        else:
                            arxiv_url = ""
                            arxiv_id = ""

                        # Published (best effort parse)
                        published_elem = entry.find("atom:published", ns)
                        published_dt = None
                        if published_elem is not None:
                            published_str = _s(published_elem.text)
                            if published_str:
                                try:
                                    published_dt = datetime.fromisoformat(published_str.replace("Z", "+00:00"))
                                except Exception:
                                    published_dt = None

                        # Authors
                        authors = []
                        for author_elem in entry.findall("atom:author", ns):
                            name_elem = author_elem.find("atom:name", ns)
                            if name_elem is not None:
                                name = _s(name_elem.text)
                                if name:
                                    authors.append(name)

                        # DOI (optional)
                        doi_elem = entry.find("arxiv:doi", ns)
                        doi = _s(doi_elem.text) if doi_elem is not None else ""

                        # Build record
                        paper["title"] = title
                        paper["abstract"] = abstract
                        paper["arxiv_id"] = arxiv_id
                        paper["url"] = arxiv_url
                        paper["published"] = published_dt
                        paper["authors"] = authors
                        paper["doi"] = doi
                        paper["source"] = "arxiv"
                        paper["category"] = category

                        # Minimum validity
                        if not paper["title"] or not paper["arxiv_id"]:
                            logger.debug(f"Skipping invalid arXiv entry (missing title/id): {paper.get('arxiv_id','')}")
                            continue

                        # Extra safety: time window filter (published_dt may be earlier than submittedDate)
                        if published_dt is not None:
                            # keep if within window (inclusive-ish)
                            if published_dt < start_time or published_dt > end_time:
                                # arXiv "published" may differ; do not drop aggressively if you don't want.
                                # Here we keep it permissive: only drop if it’s wildly outside by > 30 days etc.
                                pass

                        # ---------------------------
                        # Relevance filtering HERE
                        # ---------------------------
                        try:
                            # Title + abstract + category can help
                            combined_text = f"{title} {abstract} {category}"
                            if "paper_is_relevant" in globals() and callable(globals()["paper_is_relevant"]):
                                if not paper_is_relevant(combined_text, scanner_config):
                                    logger.debug(f"Filtered out (irrelevant): {title[:80]}...")
                                    continue
                        except Exception as e:
                            # If relevance filter fails, do not kill the pipeline
                            logger.warning(f"Relevance filter error (kept paper): {e}")

                        all_papers.append(paper)
                        fetched_for_cat += 1
                        added_this_page += 1

                        if fetched_for_cat >= max_papers:
                            break

                    except Exception as e:
                        logger.warning(f"Error parsing arXiv entry: {e}")
                        continue

                logger.info(f"Added {added_this_page} relevant papers (category={category}, total_for_cat={fetched_for_cat})")

                # Pagination: next page
                start_offset += len(entries)

                # If this page added nothing AND entries were small, likely end / irrelevant only
                if len(entries) < page_size:
                    break

                # Rate limiting: 3 seconds between requests
                time.sleep(3)

            except requests.exceptions.RequestException as e:
                logger.error(f"arXiv API request failed for category {category}: {e}")
                break
            except ET.ParseError as e:
                logger.error(f"XML parsing failed for category {category}: {e}")
                break
            except Exception as e:
                logger.error(f"Unexpected error fetching arXiv papers for category {category}: {e}")
                break

        # Between categories
        if categories.index(category) < len(categories) - 1:
            time.sleep(3)

    logger.info(f"Total arXiv papers fetched (after relevance filter): {len(all_papers)}")
    return all_papers


# --- Execute fetch ---
logger.info("Starting arXiv paper fetch")

arxiv_papers = fetch_arxiv_papers(
    start_time=scan_window["start_time"],
    end_time=scan_window["end_time"],
    categories=scanner_config.get("arxiv_categories", []),
    max_papers=scanner_config.get("max_papers_per_source", 100),
)

logger.info(f"Fetched {len(arxiv_papers)} papers from arXiv (after relevance filter)")

# Log sample paper for verification
if arxiv_papers:
    sample = arxiv_papers[0]
    logger.info("Sample arXiv paper:")
    logger.info(f"  Title: {sample['title'][:80]}...")
    logger.info(f"  arXiv ID: {sample.get('arxiv_id', 'N/A')}")
    logger.info(f"  DOI: {sample.get('doi', 'N/A')}")
    logger.info(f"  Authors: {', '.join(sample.get('authors', [])[:3])}...")
    logger.info(f"  Published: {sample.get('published', 'N/A')}")
    logger.info(f"  Abstract: {sample.get('abstract', '')[:100]}...")
    logger.info(f"  Category: {sample.get('category', 'N/A')}")
else:
    logger.warning("No papers fetched from arXiv (may indicate API issue, empty result set, or strict relevance filter)")

logger.info("Cell 06 complete: arXiv papers fetched (with relevance filtering)")


2026-01-25 14:13:47 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 | Starting arXiv paper fetch
2026-01-25 14:13:47 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 | Fetching arXiv papers for category: econ.GN
2026-01-25 14:13:47 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 | arXiv API request: cat=econ.GN, start=0, max=100
2026-01-25 14:13:48 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 | No more results for category econ.GN (start=0)
2026-01-25 14:13:51 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 | Fetching arXiv papers for category: cs.CY
2026-01-25 14:13:51 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 | arXiv API request: cat=cs.CY, start=0, max=100
2026-01-25 14:13:51 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 | No more results for category cs.CY (start=0)
2026-01-25 14:13:54 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 | Fetching arXiv papers for category: physics.soc-ph
2026-01-25 14:13:54 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 | 

In [39]:
# ============================================================
# Cell 07 — Fetch papers from Semantic Scholar API (FULL REPLACEMENT)
# ============================================================
# Adds:
#   - Theme-keyword-driven query (instead of naive "query=field")
#   - Relevance filtering via paper_is_relevant() using scanner_config
#   - 429 handling (Retry-After + exponential backoff + jitter)
#   - Safe parsing (None-safe strings)
#   - Conservative request sizing (limit <= 20 by default to reduce rate-limit risk)
#   - Graceful partial failures (per-field isolation)
#
# Requires (recommended in an earlier cell, e.g., Cell 03):
#   - paper_is_relevant(text: str, config: dict) -> bool
#
# Config expectations:
#   scanner_config may include:
#     - semantic_scholar_fields: List[str]
#     - max_papers_per_source: int
#     - s2_page_limit: int (<=100, recommended 20)
#     - s2_max_retries: int
#     - s2_min_sleep_sec: float
#     - s2_field_sleep_sec: float
#     - theme_keywords: Dict[str, List[str]]  (preferred)
#         OR global THEMES: Dict[str, List[str]] is also accepted
#     - negative_keywords: List[str] (optional)
# ============================================================

from datetime import datetime, timezone
from typing import Any, Dict, List, Optional
import os
import time
import random
import requests

# ----------------------------
# Helpers
# ----------------------------
def _s(value: Any, default: str = "") -> str:
    """Safe string conversion (None-safe)."""
    if value is None:
        return default
    return str(value).strip()

def _parse_pub_date(pub_date_str: str) -> Optional[datetime]:
    """Parse YYYY-MM-DD to UTC datetime; return None if invalid/empty."""
    s = _s(pub_date_str)
    if not s:
        return None
    try:
        return datetime.strptime(s, "%Y-%m-%d").replace(tzinfo=timezone.utc)
    except Exception:
        return None

def _request_with_backoff(
    url: str,
    params: Dict[str, Any],
    headers: Dict[str, str],
    timeout: int = 30,
    max_retries: int = 6,
    base_backoff: float = 1.0,
    max_backoff: float = 60.0,
) -> requests.Response:
    """
    Request wrapper with:
      - 429 handling (Retry-After if present)
      - exponential backoff + jitter for transient errors
    """
    last_err: Optional[Exception] = None

    for attempt in range(max_retries):
        try:
            resp = requests.get(url, params=params, headers=headers, timeout=timeout)

            # 429: rate limit
            if resp.status_code == 429:
                retry_after = resp.headers.get("Retry-After")
                if retry_after:
                    try:
                        wait = float(retry_after)
                    except Exception:
                        wait = min(max_backoff, base_backoff * (2 ** attempt))
                else:
                    wait = min(max_backoff, base_backoff * (2 ** attempt))

                wait = wait + random.random()  # jitter
                logger.warning(
                    f"S2 rate-limited (429). Sleeping {wait:.1f}s then retrying "
                    f"(attempt {attempt+1}/{max_retries})..."
                )
                time.sleep(wait)
                continue

            resp.raise_for_status()
            return resp

        except requests.exceptions.RequestException as e:
            last_err = e
            wait = min(max_backoff, base_backoff * (2 ** attempt)) + random.random()
            logger.warning(
                f"S2 request error: {e}. Sleeping {wait:.1f}s then retrying "
                f"(attempt {attempt+1}/{max_retries})..."
            )
            time.sleep(wait)

    raise RuntimeError(f"S2 request failed after {max_retries} retries. Last error: {last_err}")

def _get_theme_keywords(config: dict) -> Dict[str, List[str]]:
    """
    Priority:
      1) scanner_config['theme_keywords']
      2) global THEMES (if exists)
      3) fallback empty dict
    """
    try:
        tk = config.get("theme_keywords")
        if isinstance(tk, dict) and tk:
            return tk
    except Exception:
        pass

    try:
        if "THEMES" in globals() and isinstance(globals()["THEMES"], dict):
            return globals()["THEMES"]
    except Exception:
        pass

    return {}

def _build_s2_query_from_themes(theme_keywords: Dict[str, List[str]], max_terms: int = 18) -> str:
    """
    Build an OR query from theme keywords, capped to avoid overly long queries.
    - We include up to max_terms terms total (across all groups).
    - Quote multi-word terms.
    """
    terms: List[str] = []
    for _, kws in theme_keywords.items():
        if not isinstance(kws, list):
            continue
        for kw in kws:
            kw = _s(kw)
            if not kw:
                continue
            # normalize tiny variants
            # (keep as-is; paper_is_relevant will do the real gating)
            if " " in kw:
                terms.append(f"\"{kw}\"")
            else:
                terms.append(kw)

    # de-dup while preserving order
    seen = set()
    uniq = []
    for t in terms:
        if t.lower() in seen:
            continue
        seen.add(t.lower())
        uniq.append(t)

    uniq = uniq[:max_terms]
    if not uniq:
        # fallback: still use something rather than empty query
        return "venture capital OR startup OR entrepreneurship"

    return " OR ".join(uniq)

# ----------------------------
# Main fetcher
# ----------------------------
def fetch_semantic_scholar_papers(
    start_time: datetime,
    end_time: datetime,
    fields_of_study: List[str],
    max_papers: int = 100,
    page_limit: int = 20,
    max_retries: int = 6,
    min_sleep_sec: float = 1.5,
    field_sleep_sec: float = 2.0,
) -> List[Dict[str, Any]]:
    """
    Fetch recent papers from Semantic Scholar API within the specified time window and fields.

    Key change vs naive approach:
      - query is built from theme keywords, not from the field string itself.
      - final selection is gated by paper_is_relevant().

    Returns list of paper dicts:
      title, authors, abstract, doi, published, source, url, s2_id,
      field_of_study, fields_of_study, arxiv_id
    """
    base_url = "https://api.semanticscholar.org/graph/v1/paper/search"

    api_key = _s(os.getenv("SEMANTIC_SCHOLAR_API_KEY", ""))
    headers = {"x-api-key": api_key} if api_key else {}

    # S2 filter is date-level; use YYYY-MM-DD
    start_date = start_time.strftime("%Y-%m-%d")
    end_date = end_time.strftime("%Y-%m-%d")

    api_fields = "paperId,title,abstract,authors,publicationDate,externalIds,url,fieldsOfStudy"

    all_papers: List[Dict[str, Any]] = []

    if not fields_of_study:
        logger.warning("No Semantic Scholar fields configured; skipping S2 fetch.")
        return all_papers

    # clamp page_limit
    page_limit = int(max(1, min(int(page_limit), 100)))

    # Build query from themes
    theme_keywords = _get_theme_keywords(scanner_config)
    query_string = _build_s2_query_from_themes(theme_keywords, max_terms=int(scanner_config.get("s2_query_max_terms", 18)))
    logger.info(f"S2 query (from themes): {query_string[:160]}{'...' if len(query_string) > 160 else ''}")

    # Optional negative keywords (we’ll mainly rely on paper_is_relevant)
    negative_keywords = scanner_config.get("negative_keywords", [])
    if not isinstance(negative_keywords, list):
        negative_keywords = []

    def _passes_negative_filter(title: str, abstract: str) -> bool:
        if not negative_keywords:
            return True
        text = f"{title} {abstract}".lower()
        for nk in negative_keywords:
            nk = _s(nk).lower()
            if nk and nk in text:
                return False
        return True

    for idx, field in enumerate(fields_of_study):
        field = _s(field)
        if not field:
            continue

        logger.info(f"Fetching Semantic Scholar papers for field: {field}")

        params: Dict[str, Any] = {
            "query": query_string,
            "fields": api_fields,
            "publicationDateOrYear": f"{start_date}:{end_date}",
            "fieldsOfStudy": field,
            "limit": page_limit,
            "offset": 0,
        }

        fetched_for_field = 0

        while fetched_for_field < max_papers:
            remaining = max_papers - fetched_for_field
            params["limit"] = min(page_limit, remaining)

            try:
                logger.info(f"S2 API request: field={field}, offset={params['offset']}, limit={params['limit']}")

                resp = _request_with_backoff(
                    base_url,
                    params=params,
                    headers=headers,
                    timeout=30,
                    max_retries=max_retries,
                    base_backoff=1.0,
                    max_backoff=60.0,
                )

                data = resp.json() if resp.content else {}
                batch = data.get("data") or []
                total = data.get("total", None)

                if not batch:
                    logger.info(f"No more results for field {field}")
                    break

                logger.info(
                    f"Retrieved {len(batch)} papers"
                    + (f" (total available: {total})" if total is not None else "")
                )

                added_this_page = 0

                for paper_data in batch:
                    try:
                        s2_id = _s(paper_data.get("paperId"))
                        if not s2_id:
                            continue

                        title = _s(paper_data.get("title"))
                        abstract = _s(paper_data.get("abstract"))

                        # cheap negative filter (optional)
                        if not _passes_negative_filter(title, abstract):
                            continue

                        # Authors
                        authors_list = []
                        for a in (paper_data.get("authors") or []):
                            if isinstance(a, dict) and a.get("name"):
                                authors_list.append(_s(a.get("name")))

                        external_ids = paper_data.get("externalIds") or {}
                        doi = _s(external_ids.get("DOI"))
                        arxiv_id = _s(external_ids.get("ArXiv"))

                        url = _s(paper_data.get("url"))
                        if not url:
                            url = f"https://www.semanticscholar.org/paper/{s2_id}"

                        paper: Dict[str, Any] = {
                            "title": title,
                            "abstract": abstract,
                            "authors": authors_list,
                            "published": _parse_pub_date(paper_data.get("publicationDate")),
                            "doi": doi,
                            "arxiv_id": arxiv_id,
                            "s2_id": s2_id,
                            "url": url,
                            "source": "semantic_scholar",
                            "field_of_study": field,
                            "fields_of_study": paper_data.get("fieldsOfStudy") or [],
                        }

                        # Require at least title (S2 sometimes returns empty titles)
                        if not paper["title"]:
                            continue

                        # ---------------------------
                        # Relevance filtering HERE
                        # ---------------------------
                        try:
                            combined_text = f"{paper['title']} {paper['abstract']} {field} " \
                                            f"{' '.join(paper.get('fields_of_study') or [])}"
                            if "paper_is_relevant" in globals() and callable(globals()["paper_is_relevant"]):
                                if not paper_is_relevant(combined_text, scanner_config):
                                    continue
                        except Exception as e:
                            # If relevance filter fails, keep paper (do not kill pipeline)
                            logger.warning(f"Relevance filter error (kept paper): {e}")

                        all_papers.append(paper)
                        fetched_for_field += 1
                        added_this_page += 1

                        if fetched_for_field >= max_papers:
                            break

                    except Exception as e:
                        logger.warning(f"Error parsing S2 paper entry: {e}")
                        continue

                logger.info(f"Added {added_this_page} relevant papers (field={field}, total_for_field={fetched_for_field})")

                if len(batch) < int(params["limit"]):
                    break

                params["offset"] += len(batch)

                time.sleep(min_sleep_sec)

            except Exception as e:
                logger.error(f"Unexpected error fetching S2 papers for field {field}: {e}")
                break

        logger.info(f"Fetched {fetched_for_field} papers for field {field}")

        if idx < len(fields_of_study) - 1:
            time.sleep(field_sleep_sec)

    logger.info(f"Total Semantic Scholar papers fetched (after relevance filter): {len(all_papers)}")
    return all_papers

# ----------------------------
# Execute fetch
# ----------------------------
logger.info("Starting Semantic Scholar paper fetch")

fields = scanner_config.get("semantic_scholar_fields", ["Economics", "Business"])
max_papers = int(scanner_config.get("max_papers_per_source", 100))
page_limit = int(scanner_config.get("s2_page_limit", 20))
max_retries = int(scanner_config.get("s2_max_retries", 6))
min_sleep_sec = float(scanner_config.get("s2_min_sleep_sec", 1.5))
field_sleep_sec = float(scanner_config.get("s2_field_sleep_sec", 2.0))

semantic_scholar_papers = fetch_semantic_scholar_papers(
    start_time=scan_window["start_time"],
    end_time=scan_window["end_time"],
    fields_of_study=fields,
    max_papers=max_papers,
    page_limit=page_limit,
    max_retries=max_retries,
    min_sleep_sec=min_sleep_sec,
    field_sleep_sec=field_sleep_sec,
)

logger.info(f"Fetched {len(semantic_scholar_papers)} papers from Semantic Scholar (after relevance filter)")

if semantic_scholar_papers:
    sample = semantic_scholar_papers[0]
    logger.info("Sample Semantic Scholar paper:")
    logger.info(f"  Title: {sample.get('title','')[:80]}...")
    logger.info(f"  S2 ID: {sample.get('s2_id', 'N/A')}")
    logger.info(f"  DOI: {sample.get('doi', 'N/A')}")
    logger.info(f"  Authors: {', '.join(sample.get('authors', [])[:3])}...")
    logger.info(f"  Published: {sample.get('published', 'N/A')}")
    logger.info(f"  Fields: {', '.join((sample.get('fields_of_study') or [])[:3])}")
    logger.info(f"  Abstract: {sample.get('abstract', '')[:100]}...")
else:
    logger.warning("No papers fetched from Semantic Scholar (may indicate API rate limit, empty results, or strict relevance filter)")

logger.info("Cell 07 complete: Semantic Scholar papers fetched (with relevance filtering)")


2026-01-25 14:16:31 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 | Starting Semantic Scholar paper fetch
2026-01-25 14:16:31 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 | S2 query (from themes): "venture capital" OR vc OR "venture investing" OR "venture investment" OR "venture fund" OR "venture financing" OR "startup financing" OR "early-stage financing...
2026-01-25 14:16:31 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 | Fetching Semantic Scholar papers for field: Economics
2026-01-25 14:16:31 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 | S2 API request: field=Economics, offset=0, limit=20
2026-01-25 14:16:31 | WARNING  | 2ac5680b-7a63-488f-a211-1860c73ee2f8 | S2 rate-limited (429). Sleeping 1.8s then retrying (attempt 1/6)...
2026-01-25 14:16:33 | WARNING  | 2ac5680b-7a63-488f-a211-1860c73ee2f8 | S2 rate-limited (429). Sleeping 2.5s then retrying (attempt 2/6)...
2026-01-25 14:16:36 | WARNING  | 2ac5680b-7a63-488f-a211-1860c73ee2f8 | S2 rate-limited (429). Sleep

In [40]:
# ============================================================
# Cell 08 — Merge and deduplicate fetched batch (FULL REPLACEMENT)
# ============================================================
# Fixes:
#   - Avoids KeyError: uses scanner_config.get("title_similarity_threshold", 0.85)
#     instead of config['title_similarity_threshold']
#   - Handles empty sources safely
#   - Keeps deterministic behavior with stable source_priority
#
# Inputs:
#   - arxiv_papers (Cell 06)
#   - semantic_scholar_papers (Cell 07)
#   - batch_add_dedup_keys (Cell 04)
#   - batch_find_similar_titles (Cell 05)
#   - scanner_config dict (Cell 01)
#
# Outputs:
#   - unique_papers: list of deduplicated papers (each has dedup_key / dedup_method)
#   - dedup_stats: summary dict
# ============================================================

logger.info("Merging papers from all sources")

all_fetched_papers: List[Dict[str, Any]] = []

# Priority: lower is better
# NOTE: If later you add OpenAlex, you can decide where it belongs.
SOURCE_PRIORITY = {
    "arxiv": 1,
    "openalex": 2,
    "semantic_scholar": 3,
}

# --- Add arXiv papers ---
for p in (arxiv_papers or []):
    p["source"] = p.get("source") or "arxiv"
    p["source_priority"] = SOURCE_PRIORITY.get(p["source"], 99)
    all_fetched_papers.append(p)

# --- Add Semantic Scholar papers ---
for p in (semantic_scholar_papers or []):
    p["source"] = p.get("source") or "semantic_scholar"
    p["source_priority"] = SOURCE_PRIORITY.get(p["source"], 99)
    all_fetched_papers.append(p)

logger.info(f"Total papers before deduplication: {len(all_fetched_papers)}")
logger.info(f"  arXiv: {len(arxiv_papers or [])}")
logger.info(f"  Semantic Scholar: {len(semantic_scholar_papers or [])}")

# --- Add dedup keys ---
logger.info("Generating dedup keys for all papers")
batch_add_dedup_keys(all_fetched_papers)

# Count by dedup method
dedup_method_counts: Dict[str, int] = {}
for p in all_fetched_papers:
    m = p.get("dedup_method") or "unknown"
    dedup_method_counts[m] = dedup_method_counts.get(m, 0) + 1
logger.info(f"Dedup key methods: {dedup_method_counts}")

# --- Exact dedup by dedup_key ---
logger.info("Building dedup key index")
dedup_key_index: Dict[str, List[int]] = {}

for idx, p in enumerate(all_fetched_papers):
    k = p.get("dedup_key")
    if not k:
        continue
    dedup_key_index.setdefault(k, []).append(idx)

exact_duplicate_groups = {k: v for k, v in dedup_key_index.items() if len(v) > 1}
logger.info(f"Found {len(exact_duplicate_groups)} exact duplicate groups (via dedup_key)")

logger.info("Resolving exact duplicates by source priority")

keep_indices = set(range(len(all_fetched_papers)))
exact_duplicates_removed = 0

for k, indices in exact_duplicate_groups.items():
    # sort by (priority, stable idx)
    indices_sorted = sorted(
        indices,
        key=lambda i: (all_fetched_papers[i].get("source_priority", 99), i),
    )
    keep_idx = indices_sorted[0]
    for ridx in indices_sorted[1:]:
        if ridx in keep_indices:
            keep_indices.discard(ridx)
            exact_duplicates_removed += 1

logger.info(f"Removed {exact_duplicates_removed} exact duplicates")
logger.info(f"Papers remaining after exact dedup: {len(keep_indices)}")

papers_after_exact_dedup = [all_fetched_papers[i] for i in sorted(keep_indices)]

# --- Title similarity dedup (fuzzy) ---
# IMPORTANT: Use scanner_config, not config.yaml keys.
threshold = float(scanner_config.get("title_similarity_threshold", 0.85))

logger.info("Performing title similarity deduplication")
logger.info(f"Title similarity threshold: {threshold:.3f}")

similarity_results = batch_find_similar_titles(
    papers_after_exact_dedup,
    threshold=threshold
)

similar_pairs = similarity_results.get("duplicates") or []
logger.info(f"Found {len(similar_pairs)} similar title pairs")

# Union-Find for transitive closure
parent = {i: i for i in range(len(papers_after_exact_dedup))}

def _find(x: int) -> int:
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

def _union(x: int, y: int) -> None:
    rx, ry = _find(x), _find(y)
    if rx == ry:
        return

    # Keep root with better source_priority; tie-break by index
    px = papers_after_exact_dedup[rx].get("source_priority", 99)
    py = papers_after_exact_dedup[ry].get("source_priority", 99)

    if (px, rx) <= (py, ry):
        parent[ry] = rx
    else:
        parent[rx] = ry

for i, j, score in similar_pairs:
    _union(i, j)

# Group by representative
groups: Dict[int, List[int]] = {}
for i in range(len(papers_after_exact_dedup)):
    r = _find(i)
    groups.setdefault(r, []).append(i)

# Select one per group (the representative)
unique_papers: List[Dict[str, Any]] = []
title_duplicates_removed = 0

for r, members in groups.items():
    unique_papers.append(papers_after_exact_dedup[r])
    if len(members) > 1:
        title_duplicates_removed += (len(members) - 1)

logger.info(f"Removed {title_duplicates_removed} title-similar duplicates")
logger.info(f"Final unique papers after all deduplication: {len(unique_papers)}")

# --- Stats summary ---
total_fetched = len(all_fetched_papers)
total_duplicates_removed = exact_duplicates_removed + title_duplicates_removed
deduplication_rate = (total_duplicates_removed / total_fetched * 100) if total_fetched else 0.0

dedup_stats = {
    "total_fetched": total_fetched,
    "exact_duplicates_removed": exact_duplicates_removed,
    "title_duplicates_removed": title_duplicates_removed,
    "total_duplicates_removed": total_duplicates_removed,
    "unique_papers": len(unique_papers),
    "deduplication_rate_pct": round(deduplication_rate, 2),
    "title_similarity_threshold": threshold,
}

logger.info("Batch deduplication complete:")
logger.info(f"  Total fetched:            {dedup_stats['total_fetched']}")
logger.info(f"  Exact duplicates removed: {dedup_stats['exact_duplicates_removed']}")
logger.info(f"  Title duplicates removed: {dedup_stats['title_duplicates_removed']}")
logger.info(f"  Unique papers:            {dedup_stats['unique_papers']}")
logger.info(f"  Deduplication rate:       {dedup_stats['deduplication_rate_pct']}%")

# --- Log samples ---
if unique_papers:
    logger.info("Sample unique papers after deduplication:")
    for i, p in enumerate(unique_papers[:3]):
        logger.info(f"  [{i}] {p.get('title','')[:70]}...")
        logger.info(f"      Source: {p.get('source','?')} | Method: {p.get('dedup_method','?')}")
        logger.info(f"      DOI: {p.get('doi','N/A')} | arXiv: {p.get('arxiv_id','N/A')}")
else:
    logger.warning("No unique papers remaining after deduplication")

logger.info("Cell 08 complete: batch merged and deduplicated")


2026-01-25 14:17:07 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 | Merging papers from all sources
2026-01-25 14:17:07 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 | Total papers before deduplication: 0
2026-01-25 14:17:07 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 |   arXiv: 0
2026-01-25 14:17:07 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 |   Semantic Scholar: 0
2026-01-25 14:17:07 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 | Generating dedup keys for all papers
2026-01-25 14:17:07 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 | Dedup key generation complete: {'doi': 0, 'arxiv_id': 0, 'title': 0, 'failed': 0}
2026-01-25 14:17:07 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 | Dedup key methods: {}
2026-01-25 14:17:07 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 | Building dedup key index
2026-01-25 14:17:07 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 | Found 0 exact duplicate groups (via dedup_key)
2026-01-25 14:17:07 | INFO     | 2ac5

In [41]:
# ============================================================
# Cell 09 — Check Notion DB for existing duplicates (FULL REPLACEMENT)
# ============================================================
# Fixes:
#   - No more NameError: find_duplicate_by_doi / find_duplicate_by_title
#     → uses 029 exports that actually exist:
#        - query_paper_by_dedup_key
#        - get_recent_papers
#        - check_duplicate_paper (optional; if present)
#        - titles_are_similar (Cell 05)
#   - No more KeyError: config['title_similarity_threshold']
#     → uses scanner_config.get(..., 0.85)
#
# Strategy (robust + cheap):
#   1) Exact dedup by Dedup Key (fast): query_paper_by_dedup_key(paper['dedup_key'])
#   2) If not found and you want fuzzy: pull recent papers once, then title-similarity scan in-memory
#
# Inputs:
#   - unique_papers (Cell 08 output)
#   - scanner_config (Cell 01)
#   - query_paper_by_dedup_key, get_recent_papers (from 029)
#   - titles_are_similar (Cell 05)
#
# Outputs:
#   - new_papers
#   - duplicate_papers
#   - notion_duplicate_stats
# ============================================================

logger.info("Starting Notion DB duplicate checking")

threshold = float(scanner_config.get("title_similarity_threshold", 0.85))
use_title_fallback = bool(scanner_config.get("use_title_similarity_fallback", True))

# How many existing papers to compare against for title similarity.
# (Tune: 300-1500; larger = safer but more API + slower)
recent_limit = int(scanner_config.get("notion_recent_papers_limit", 800))

notion_duplicate_stats = {
    "total_checked": len(unique_papers),
    "duplicates_found": 0,
    "duplicates_by_dedup_key": 0,
    "duplicates_by_title": 0,
    "check_errors": 0,
    "new_papers": 0,
    "title_similarity_threshold": threshold,
    "title_fallback_enabled": use_title_fallback,
    "recent_compare_pool": 0,
}

new_papers: List[Dict[str, Any]] = []
duplicate_papers: List[Dict[str, Any]] = []  # {paper, notion_page_id, match_method, score?}

# ---------------------------------------------------------------------
# (Optional) Load recent Notion papers once for title-similarity fallback
# ---------------------------------------------------------------------
recent_papers_cache: List[Dict[str, Any]] = []
if use_title_fallback:
    try:
        logger.info(f"Loading recent papers from Notion for title fallback (limit={recent_limit})")
        # 029: get_recent_papers should return list of dicts (include page_id + title at minimum)
        recent_papers_cache = get_recent_papers(limit=recent_limit)
        notion_duplicate_stats["recent_compare_pool"] = len(recent_papers_cache)
        logger.info(f"Loaded {len(recent_papers_cache)} recent papers for title comparison")
    except Exception as e:
        logger.warning(f"Failed to load recent papers for title fallback: {e}")
        notion_duplicate_stats["check_errors"] += 1
        recent_papers_cache = []

# Helper: best-effort extract title + page_id from 029 shapes
def _extract_notion_title(rec: Dict[str, Any]) -> str:
    # support multiple shapes
    for k in ["title", "name", "Name"]:
        v = rec.get(k)
        if isinstance(v, str) and v.strip():
            return v.strip()
    # sometimes nested
    t = rec.get("properties", {}).get("Name") if isinstance(rec.get("properties"), dict) else None
    if isinstance(t, str) and t.strip():
        return t.strip()
    return ""

def _extract_page_id(rec: Dict[str, Any]) -> Optional[str]:
    for k in ["page_id", "id", "pageId"]:
        v = rec.get(k)
        if isinstance(v, str) and v.strip():
            return v.strip()
    return None

# ---------------------------------------------------------------------
# Main loop
# ---------------------------------------------------------------------
for idx, paper in enumerate(unique_papers):
    if idx > 0 and idx % 10 == 0:
        logger.info(f"Checked {idx}/{len(unique_papers)} papers...")

    title = (paper.get("title") or "").strip()
    dedup_key = paper.get("dedup_key")
    duplicate_found = False
    duplicate_page_id: Optional[str] = None
    match_method: Optional[str] = None
    match_score: Optional[float] = None

    try:
        # ---------------------------------------------------------
        # 1) Exact match via Dedup Key (preferred)
        # ---------------------------------------------------------
        if dedup_key:
            try:
                existing = query_paper_by_dedup_key(dedup_key)
                if existing:
                    duplicate_found = True
                    duplicate_page_id = _extract_page_id(existing) or existing.get("page_id")
                    match_method = "dedup_key"
                    notion_duplicate_stats["duplicates_by_dedup_key"] += 1
            except Exception as e:
                logger.warning(f"Error checking dedup_key for '{title[:60]}...': {e}")
                notion_duplicate_stats["check_errors"] += 1

        # ---------------------------------------------------------
        # 2) Title similarity fallback (in-memory against recent pool)
        # ---------------------------------------------------------
        if (not duplicate_found) and use_title_fallback and title and recent_papers_cache:
            best = None  # (score, page_id, notion_title)
            for rec in recent_papers_cache:
                nt = _extract_notion_title(rec)
                if not nt:
                    continue
                is_sim, score = titles_are_similar(title, nt, threshold=threshold)
                if not is_sim:
                    continue
                pid = _extract_page_id(rec)
                if not pid:
                    continue
                if (best is None) or (score > best[0]):
                    best = (score, pid, nt)

            if best is not None:
                duplicate_found = True
                match_method = "title_similarity"
                match_score = float(best[0])
                duplicate_page_id = best[1]
                notion_duplicate_stats["duplicates_by_title"] += 1

        # ---------------------------------------------------------
        # Classify
        # ---------------------------------------------------------
        if duplicate_found:
            notion_duplicate_stats["duplicates_found"] += 1
            duplicate_papers.append(
                {
                    "paper": paper,
                    "notion_page_id": duplicate_page_id,
                    "match_method": match_method,
                    "score": match_score,
                }
            )
            logger.info(
                f"Duplicate #{notion_duplicate_stats['duplicates_found']}: "
                f"'{title[:60]}...' (method={match_method}, page={duplicate_page_id}"
                + (f", score={match_score:.3f}" if match_score is not None else "")
                + ")"
            )
        else:
            new_papers.append(paper)
            notion_duplicate_stats["new_papers"] += 1

        # Rate limiting (query_paper_by_dedup_key is a Notion call)
        time.sleep(0.2)

    except Exception as e:
        logger.error(f"Unexpected error checking paper '{title[:60]}...': {e}")
        notion_duplicate_stats["check_errors"] += 1
        # Conservative: treat as new (so you don't miss papers)
        new_papers.append(paper)
        notion_duplicate_stats["new_papers"] += 1

# ---------------------------------------------------------------------
# Summary
# ---------------------------------------------------------------------
logger.info("Notion DB duplicate checking complete:")
logger.info(f"  Total checked:     {notion_duplicate_stats['total_checked']}")
logger.info(f"  Duplicates found:  {notion_duplicate_stats['duplicates_found']}")
logger.info(f"    By dedup_key:    {notion_duplicate_stats['duplicates_by_dedup_key']}")
logger.info(f"    By title sim:    {notion_duplicate_stats['duplicates_by_title']}")
logger.info(f"  New papers:        {notion_duplicate_stats['new_papers']}")
logger.info(f"  Check errors:      {notion_duplicate_stats['check_errors']}")
logger.info(f"  Title threshold:   {threshold:.3f}")
logger.info(f"  Title fallback:    {use_title_fallback}")
logger.info(f"  Compare pool size: {notion_duplicate_stats['recent_compare_pool']}")

dup_rate = (
    notion_duplicate_stats["duplicates_found"] / notion_duplicate_stats["total_checked"] * 100
    if notion_duplicate_stats["total_checked"] else 0.0
)
logger.info(f"  Notion duplicate rate: {dup_rate:.1f}%")

# Samples
if new_papers:
    logger.info(f"Sample new papers (first 3 of {len(new_papers)}):")
    for i, p in enumerate(new_papers[:3]):
        logger.info(f"  [{i+1}] {p.get('title','')[:70]}...")
        logger.info(f"      Source: {p.get('source','?')}")
        logger.info(f"      DOI: {p.get('doi','N/A')}, arXiv: {p.get('arxiv_id','N/A')}")
        dk = p.get("dedup_key") or ""
        logger.info(f"      Dedup key: {dk[:16]}... (method: {p.get('dedup_method','?')})")

if duplicate_papers:
    logger.info(f"Sample duplicates found (first 3 of {len(duplicate_papers)}):")
    for i, d in enumerate(duplicate_papers[:3]):
        p = d["paper"]
        logger.info(f"  [{i+1}] {p.get('title','')[:70]}...")
        logger.info(f"      Method: {d.get('match_method')}, Page: {d.get('notion_page_id')}"
                    + (f", Score: {d.get('score'):.3f}" if d.get("score") is not None else ""))

logger.info("Cell 09 complete: Notion duplicate checking finished")


2026-01-25 14:17:09 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 | Starting Notion DB duplicate checking
2026-01-25 14:17:09 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 | Loading recent papers from Notion for title fallback (limit=800)
2026-01-25 14:17:09 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 | Introspected 20 properties from database 2a98e0e4d16280cbb6cbdcd1ebedee54
2026-01-25 14:17:10 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 | Loaded 70 recent papers for title comparison
2026-01-25 14:17:10 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 | Notion DB duplicate checking complete:
2026-01-25 14:17:10 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 |   Total checked:     0
2026-01-25 14:17:10 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 |   Duplicates found:  0
2026-01-25 14:17:10 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 |     By dedup_key:    0
2026-01-25 14:17:10 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 |     By title sim:    0

In [42]:
# ============================================================
# Cell 10 — Generate short summaries (FULL REPLACEMENT)
# ============================================================
# Fixes:
#   - SyntaxError: "exse:" -> proper else
#   - Safer defaults if llm_provider/llm_model/llm_temperature are missing
#   - Handles OpenAI SDK presence/absence gracefully
# ============================================================

logger.info("Starting summary generation for new papers")

# --- Safe defaults (in case upstream didn't define them) ---
llm_provider = globals().get("llm_provider", "OpenAI")
llm_model = globals().get("llm_model", "gpt-4o-mini")
llm_temperature = globals().get("llm_temperature", 0.2)

summary_stats = {
    "total_papers": len(new_papers),
    "llm_summaries": 0,
    "fallback_summaries": 0,
    "summary_errors": 0,
    "llm_tokens_used": 0,
}

# --- Check LLM availability ---
llm_available = False
openai_client = None

if llm_provider == "OpenAI":
    try:
        import openai  # type: ignore
        openai_api_key = os.getenv("OPENAI_API_KEY", "")
        if openai_api_key:
            # New-style client (openai>=1.x)
            try:
                openai_client = openai.OpenAI(api_key=openai_api_key)
                llm_available = True
            except Exception:
                # Fallback: old-style (openai<1.x) - we will handle later
                openai_client = openai
                openai.api_key = openai_api_key
                llm_available = True

            logger.info(f"LLM available: {llm_provider} ({llm_model})")
        else:
            logger.warning("OPENAI_API_KEY not set; using fallback summaries")
    except ImportError:
        logger.warning("OpenAI package not installed; using fallback summaries")
else:
    logger.warning(f"Unsupported LLM provider '{llm_provider}'; using fallback summaries")

if not llm_available:
    logger.info("LLM summarization unavailable; all summaries will use abstract truncation fallback")

# --- Prompt template ---
SUMMARY_PROMPT_TEMPLATE = """You are a research assistant helping a venture capital firm monitor academic literature.

Generate a concise 2-3 sentence summary of the following paper. Focus on:
- Key findings or contributions
- Relevance to venture capital, startups, entrepreneurship, or innovation policy
- Practical implications (if any)

Paper title: {title}

Abstract:
{abstract}

Provide ONLY the summary (2-3 sentences, 50-100 words). No preamble.
"""


def generate_llm_summary(paper: Dict[str, Any]) -> Optional[str]:
    """Generate summary using OpenAI API. Returns None on any failure."""
    if not llm_available or not openai_client:
        return None

    title = (paper.get("title") or "").strip()
    abstract = (paper.get("abstract") or "").strip()
    if not abstract:
        return None

    prompt = SUMMARY_PROMPT_TEMPLATE.format(
        title=title,
        abstract=abstract[:2000],
    )

    try:
        # New-style client: openai.OpenAI().chat.completions.create
        if hasattr(openai_client, "chat") and hasattr(openai_client.chat, "completions"):
            resp = openai_client.chat.completions.create(
                model=llm_model,
                messages=[
                    {"role": "system", "content": "You are a concise research summarization assistant."},
                    {"role": "user", "content": prompt},
                ],
                temperature=llm_temperature,
                max_tokens=180,
                timeout=30,
            )
            summary = resp.choices[0].message.content.strip()

            if getattr(resp, "usage", None) is not None:
                summary_stats["llm_tokens_used"] += getattr(resp.usage, "total_tokens", 0) or 0

            return summary

        # Old-style: openai.ChatCompletion.create
        if hasattr(openai_client, "ChatCompletion"):
            resp = openai_client.ChatCompletion.create(
                model=llm_model,
                messages=[
                    {"role": "system", "content": "You are a concise research summarization assistant."},
                    {"role": "user", "content": prompt},
                ],
                temperature=llm_temperature,
                max_tokens=180,
                request_timeout=30,
            )
            summary = resp["choices"][0]["message"]["content"].strip()
            usage = resp.get("usage", {})
            summary_stats["llm_tokens_used"] += int(usage.get("total_tokens", 0) or 0)
            return summary

        return None

    except Exception as e:
        logger.warning(f"LLM summary generation failed: {e}")
        return None


def generate_fallback_summary(paper: Dict[str, Any]) -> str:
    """Fallback summary by truncating abstract."""
    abstract = (paper.get("abstract") or "").strip()
    if not abstract:
        return "No abstract available."

    if len(abstract) <= 250:
        return abstract

    truncated = abstract[:250]
    last_period = truncated.rfind(". ")
    if last_period > 100:
        truncated = truncated[: last_period + 1]
    else:
        truncated = truncated.rstrip() + "..."
    return truncated


def add_summary_to_paper(paper: Dict[str, Any]) -> Dict[str, Any]:
    """Add paper['summary'] in-place."""
    summary = None

    if llm_available:
        summary = generate_llm_summary(paper)
        if summary:
            summary_stats["llm_summaries"] += 1

    if not summary:
        summary = generate_fallback_summary(paper)
        summary_stats["fallback_summaries"] += 1

    paper["summary"] = summary
    return paper


# --- Generate summaries ---
logger.info(f"Generating summaries for {len(new_papers)} new papers")

for idx, paper in enumerate(new_papers):
    if idx > 0 and idx % 10 == 0:
        logger.info(f"Generated {idx}/{len(new_papers)} summaries...")

    try:
        add_summary_to_paper(paper)

        # Rate limit only when we actually used LLM for this paper
        if llm_available and summary_stats["llm_summaries"] > 0:
            time.sleep(1)

    except Exception as e:
        logger.error(f"Unexpected error generating summary for paper {idx}: {e}")
        paper["summary"] = paper.get("summary") or "Summary generation failed."
        summary_stats["summary_errors"] += 1

# --- Summary stats ---
logger.info("Summary generation complete:")
logger.info(f"  Total papers:        {summary_stats['total_papers']}")
logger.info(f"  LLM summaries:       {summary_stats['llm_summaries']}")
logger.info(f"  Fallback summaries:  {summary_stats['fallback_summaries']}")
logger.info(f"  Errors:              {summary_stats['summary_errors']}")

if summary_stats["llm_tokens_used"] > 0:
    logger.info(f"  LLM tokens used:     {summary_stats['llm_tokens_used']:,}")

if summary_stats["total_papers"] > 0 and summary_stats["llm_summaries"] > 0:
    llm_rate = summary_stats["llm_summaries"] / summary_stats["total_papers"] * 100
    logger.info(f"  LLM usage rate:      {llm_rate:.1f}%")

# --- Sample summaries ---
if new_papers:
    logger.info("Sample summaries (first 3):")
    for i, p in enumerate(new_papers[:3]):
        logger.info(f"  [{i+1}] {p.get('title','')[:60]}...")
        logger.info(f"      Summary: {(p.get('summary') or '')[:120]}...")
        logger.info("")

logger.info("Cell 10 complete: summaries generated for all new papers")


2026-01-25 14:17:14 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 | Starting summary generation for new papers
2026-01-25 14:17:14 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 | LLM available: OpenAI (gpt-4o-mini)
2026-01-25 14:17:14 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 | Generating summaries for 0 new papers
2026-01-25 14:17:14 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 | Summary generation complete:
2026-01-25 14:17:14 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 |   Total papers:        0
2026-01-25 14:17:14 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 |   LLM summaries:       0
2026-01-25 14:17:14 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 |   Fallback summaries:  0
2026-01-25 14:17:14 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 |   Errors:              0
2026-01-25 14:17:14 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 | Cell 10 complete: summaries generated for all new papers


In [43]:
# ============================================================
# Cell 11 — Create new paper records in Notion (FULL REPLACEMENT v2)
# ============================================================
# Fixes:
#   - create_paper_inbox() unknown kwargs (e.g., ingested_at) -> filtered by signature
#   - dry_run resolved safely (scanner_config -> config -> default False)
#   - robust field normalization (None-safe)
# ============================================================

import inspect

logger.info("Starting Notion paper record creation")

# ---------- Safe config resolution ----------
dry_run = False
try:
    if "scanner_config" in globals() and isinstance(scanner_config, dict):
        dry_run = bool(scanner_config.get("dry_run", dry_run))
except Exception:
    pass
try:
    if "config" in globals() and isinstance(config, dict) and "dry_run" in config:
        dry_run = bool(config.get("dry_run"))
except Exception:
    pass

# ---------- Choose Notion write function ----------
create_fn = None
create_fn_name = None
if "create_paper_inbox" in globals() and callable(globals()["create_paper_inbox"]):
    create_fn = globals()["create_paper_inbox"]
    create_fn_name = "create_paper_inbox"
elif "create_paper" in globals() and callable(globals()["create_paper"]):
    create_fn = globals()["create_paper"]
    create_fn_name = "create_paper"
else:
    raise RuntimeError(
        "No Notion paper creation function found. Expected 029 to export "
        "`create_paper_inbox` (preferred) or `create_paper`."
    )

logger.info(f"Notion create function: {create_fn_name}")
logger.info(f"Dry run mode: {dry_run}")

# ---------- Inspect signature (allowed kwargs) ----------
sig = inspect.signature(create_fn)
allowed_params = set(sig.parameters.keys())

def filter_kwargs(kwargs: dict) -> dict:
    """Drop kwargs not accepted by create_fn."""
    return {k: v for k, v in kwargs.items() if k in allowed_params}

# ---------- Tracking ----------
write_stats = {
    "total_papers": len(new_papers),
    "successful_writes": 0,
    "failed_writes": 0,
    "dry_run_skipped": 0,
    "write_errors": [],
}
created_records = []  # (title, page_id)

ingested_at_dt = datetime.now(timezone.utc)
ingested_at_str = ingested_at_dt.date().isoformat()

if dry_run:
    logger.warning("DRY RUN MODE: No records will be written to Notion")
    logger.info(f"Would create {len(new_papers)} paper records")

logger.info(f"Creating {len(new_papers)} paper records in Notion Papers DB")

def _s(x):
    if x is None:
        return ""
    if not isinstance(x, str):
        x = str(x)
    return x.strip()

def _authors_to_str(v):
    if v is None:
        return ""
    if isinstance(v, list):
        vals = [_s(a) for a in v if _s(a)]
        return ", ".join(vals)
    return _s(v)

def _published_to_datestr(v):
    if isinstance(v, datetime):
        return v.strftime("%Y-%m-%d")
    if isinstance(v, str):
        p = v.strip()
        if not p:
            return ""
        try:
            dt = datetime.fromisoformat(p.replace("Z", "+00:00"))
            return dt.strftime("%Y-%m-%d")
        except Exception:
            return p[:10] if len(p) >= 10 else ""
    return ""

# ---------- Create loop ----------
for idx, paper in enumerate(new_papers):
    if idx > 0 and idx % 10 == 0:
        logger.info(f"Processed {idx}/{len(new_papers)} papers...")

    title = _s(paper.get("title"))
    paper_title_short = title[:100] if title else "UNTITLED"

    try:
        if not title:
            logger.warning(f"Skipping paper {idx}: missing title")
            write_stats["failed_writes"] += 1
            continue

        authors_year = _authors_to_str(paper.get("authors"))
        abstract = _s(paper.get("abstract"))
        summary = _s(paper.get("summary"))

        doi = _s(paper.get("doi"))
        arxiv_id = _s(paper.get("arxiv_id"))
        url = _s(paper.get("url"))
        source_uid = ("doi:" + doi) if doi else (("arxiv:" + arxiv_id) if arxiv_id else "")

        dedup_key = _s(paper.get("dedup_key"))
        pdf_status = "PENDING"

        # Dry run
        if dry_run:
            logger.info(f"[DRY RUN] Would create paper: '{title[:60]}...'")
            logger.info(f"  Authors: {authors_year[:80] if authors_year else 'N/A'}")
            logger.info(f"  DOI: {doi if doi else 'N/A'}")
            logger.info(f"  arXiv ID: {arxiv_id if arxiv_id else 'N/A'}")
            logger.info(f"  URL: {url if url else 'N/A'}")
            logger.info(f"  Dedup key: {dedup_key[:16] if dedup_key else 'N/A'}...")
            logger.info(f"  Run ID: {run_id}")
            logger.info(f"  Ingested at: {ingested_at_str}")
            logger.info("")
            write_stats["dry_run_skipped"] += 1
            continue

        # ---- Build candidate kwargs (rich) then filter by signature ----
        candidate_kwargs = dict(
            name=title,
            authors_year=authors_year,
            tags=[],
            pdf_link=url,
            status="INBOX",
            dedup_key=dedup_key,
            source_uid=source_uid,
            run_id=run_id,
            pdf_status=pdf_status,
            slide_1_url=None,
            # extras (only passed if wrapper supports)
            abstract=abstract,
            summary=summary,
            doi=doi,
            arxiv_id=arxiv_id,
            published_date=_published_to_datestr(paper.get("published")),
            ingested_at=ingested_at_str,  # ← 受け付けないなら自動で落ちる
        )

        kwargs = filter_kwargs(candidate_kwargs)

        page = create_fn(**kwargs)

        page_id = None
        if isinstance(page, dict):
            page_id = page.get("page_id") or page.get("id")
        elif isinstance(page, str):
            page_id = page

        if page_id:
            write_stats["successful_writes"] += 1
            created_records.append((title, page_id))
            logger.info(
                f"Created paper [{write_stats['successful_writes']}]: '{paper_title_short}' "
                f"(page_id: {page_id})"
            )
        else:
            write_stats["failed_writes"] += 1
            msg = f"Create returned no page_id for '{paper_title_short}'"
            write_stats["write_errors"].append(msg)
            logger.warning(msg)

        time.sleep(0.3)

    except Exception as e:
        write_stats["failed_writes"] += 1
        msg = f"Unexpected error processing paper {idx} ('{paper_title_short}'): {e}"
        write_stats["write_errors"].append(msg)
        logger.error(msg)

# ---------- Summary ----------
logger.info("Notion paper record creation complete:")
logger.info(f"  Total papers: {write_stats['total_papers']}")

if dry_run:
    logger.info(f"  Dry run mode: {write_stats['dry_run_skipped']} writes simulated (no actual writes)")
else:
    logger.info(f"  Successful writes: {write_stats['successful_writes']}")
    logger.info(f"  Failed writes: {write_stats['failed_writes']}")
    if write_stats["total_papers"] > 0:
        logger.info(f"  Success rate: {write_stats['successful_writes']/write_stats['total_papers']*100:.1f}%")
    if write_stats["write_errors"]:
        logger.warning(f"Write errors encountered: {len(write_stats['write_errors'])} total")
        logger.warning("First 3 errors:")
        for err in write_stats["write_errors"][:3]:
            logger.warning(f"  - {err}")

if created_records and not dry_run:
    logger.info(f"Sample created records (first 3 of {len(created_records)}):")
    for i, (t, pid) in enumerate(created_records[:3]):
        logger.info(f"  [{i+1}] '{t[:60]}...'")
        logger.info(f"      Page ID: {pid}")
        logger.info(f"      URL: https://www.notion.so/{pid.replace('-', '')}")

logger.info("Cell 11 complete: Notion paper records created")


2026-01-25 14:17:16 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 | Starting Notion paper record creation
2026-01-25 14:17:16 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 | Notion create function: create_paper_inbox
2026-01-25 14:17:16 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 | Dry run mode: False
2026-01-25 14:17:16 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 | Creating 0 paper records in Notion Papers DB
2026-01-25 14:17:16 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 | Notion paper record creation complete:
2026-01-25 14:17:16 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 |   Total papers: 0
2026-01-25 14:17:16 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 |   Successful writes: 0
2026-01-25 14:17:16 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 |   Failed writes: 0
2026-01-25 14:17:16 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 | Cell 11 complete: Notion paper records created


In [44]:
# ============================================================
# Cell 12 — Persist state and report statistics (FULL REPLACEMENT)
# ============================================================
# Fixes:
#   - KeyError on notion_duplicate_stats missing keys (duplicates_by_doi, etc.)
#   - config['dry_run'] missing -> safe resolution (scanner_config -> config -> default False)
#   - run_start_time source: prefer run_start_time variable; fallback to ingested_at_dt from Cell 11; fallback now()
#   - defensive stats compilation using .get with defaults
# ============================================================

logger.info("Starting Cell 12: Persist state and report statistics")

# ---------- Safe dry_run resolution ----------
dry_run = False
try:
    if "scanner_config" in globals() and isinstance(scanner_config, dict):
        dry_run = bool(scanner_config.get("dry_run", dry_run))
except Exception:
    pass
try:
    if "config" in globals() and isinstance(config, dict) and "dry_run" in config:
        dry_run = bool(config.get("dry_run"))
except Exception:
    pass

# ---------- Safe run timing ----------
run_end_time = datetime.now(timezone.utc)

# Prefer explicit run_start_time if you have it; else try Cell 11 variables; else now()
_run_start = None
if "run_start_time" in globals() and isinstance(globals()["run_start_time"], datetime):
    _run_start = globals()["run_start_time"]
elif "ingested_at_dt" in globals() and isinstance(globals()["ingested_at_dt"], datetime):
    _run_start = globals()["ingested_at_dt"]
elif "ingested_at" in globals() and isinstance(globals()["ingested_at"], datetime):
    _run_start = globals()["ingested_at"]
else:
    _run_start = run_end_time  # fallback (0 min)

elapsed_seconds = (run_end_time - _run_start).total_seconds()
elapsed_minutes = elapsed_seconds / 60.0

logger.info(f"Scan run {run_id} completed in {elapsed_minutes:.2f} minutes")

# ---------- Helpers ----------
def _get(d: dict, k: str, default=0):
    try:
        return d.get(k, default) if isinstance(d, dict) else default
    except Exception:
        return default

def _scan_window_iso(sw: dict, key: str) -> str:
    try:
        v = sw.get(key)
        if isinstance(v, datetime):
            return v.isoformat()
        if isinstance(v, str):
            return v
    except Exception:
        pass
    return ""

# ---------- Normalize Notion duplicate stats keys ----------
# Accept both naming conventions:
#   - duplicates_by_doi / duplicates_by_arxiv_id / duplicates_by_title
#   - duplicates_found_by_doi / duplicates_found_by_arxiv_id / duplicates_found_by_title
#   - duplicates_by_title_similarity (rare)
notion_stats = notion_duplicate_stats if isinstance(notion_duplicate_stats, dict) else {}

dup_by_doi = (
    _get(notion_stats, "duplicates_by_doi", None)
    if _get(notion_stats, "duplicates_by_doi", None) is not None
    else _get(notion_stats, "duplicates_found_by_doi", 0)
)

dup_by_arxiv = (
    _get(notion_stats, "duplicates_by_arxiv_id", None)
    if _get(notion_stats, "duplicates_by_arxiv_id", None) is not None
    else _get(notion_stats, "duplicates_found_by_arxiv_id", 0)
)

dup_by_title = (
    _get(notion_stats, "duplicates_by_title", None)
    if _get(notion_stats, "duplicates_by_title", None) is not None
    else _get(notion_stats, "duplicates_found_by_title", None)
    if _get(notion_stats, "duplicates_found_by_title", None) is not None
    else _get(notion_stats, "duplicates_by_title_similarity", 0)
)

# ---------- Compile comprehensive statistics (defensive) ----------
final_stats = {
    "run_id": run_id,
    "run_start": _run_start.isoformat(),
    "run_end": run_end_time.isoformat(),
    "elapsed_minutes": round(elapsed_minutes, 2),

    "scan_window": {
        "start_time": _scan_window_iso(scan_window, "start_time"),
        "end_time": _scan_window_iso(scan_window, "end_time"),
        "window_hours": round(float(_get(scan_window, "window_hours", 0.0)), 2) if isinstance(scan_window, dict) else 0.0
    },

    "fetch": {
        "arxiv_papers": len(arxiv_papers) if "arxiv_papers" in globals() and isinstance(arxiv_papers, list) else 0,
        "semantic_scholar_papers": len(semantic_scholar_papers) if "semantic_scholar_papers" in globals() and isinstance(semantic_scholar_papers, list) else 0,
        "total_fetched": _get(dedup_stats, "total_fetched", 0),
    },

    "batch_dedup": {
        "exact_duplicates_removed": _get(dedup_stats, "exact_duplicates_removed", 0),
        "title_duplicates_removed": _get(dedup_stats, "title_duplicates_removed", 0),
        "total_duplicates_removed": _get(dedup_stats, "total_duplicates_removed", 0),
        "unique_papers": _get(dedup_stats, "unique_papers", 0),
        "deduplication_rate_pct": _get(dedup_stats, "deduplication_rate_pct", 0),
    },

    "notion_check": {
        "total_checked": _get(notion_stats, "total_checked", len(unique_papers) if "unique_papers" in globals() else 0),
        "duplicates_found": _get(notion_stats, "duplicates_found", 0),
        "duplicates_by_doi": int(dup_by_doi) if dup_by_doi is not None else 0,
        "duplicates_by_arxiv_id": int(dup_by_arxiv) if dup_by_arxiv is not None else 0,
        "duplicates_by_title": int(dup_by_title) if dup_by_title is not None else 0,
        "new_papers": _get(notion_stats, "new_papers", len(new_papers) if "new_papers" in globals() else 0),
        "check_errors": _get(notion_stats, "check_errors", 0),
    },

    "summary": {
        "total_papers": _get(summary_stats, "total_papers", len(new_papers) if "new_papers" in globals() else 0),
        "llm_summaries": _get(summary_stats, "llm_summaries", 0),
        "fallback_summaries": _get(summary_stats, "fallback_summaries", 0),
        "summary_errors": _get(summary_stats, "summary_errors", 0),
        "llm_tokens_used": _get(summary_stats, "llm_tokens_used", 0),
    },

    "write": {
        "total_papers": _get(write_stats, "total_papers", len(new_papers) if "new_papers" in globals() else 0),
        "successful_writes": _get(write_stats, "successful_writes", 0),
        "failed_writes": _get(write_stats, "failed_writes", 0),
        "dry_run_skipped": _get(write_stats, "dry_run_skipped", 0),
        "dry_run_mode": dry_run,
    },
}

final_stats["pipeline"] = {
    "total_errors": int(final_stats["notion_check"]["check_errors"]) + int(final_stats["summary"]["summary_errors"]) + int(final_stats["write"]["failed_writes"]),
    "success": (final_stats["write"]["failed_writes"] == 0) or dry_run,
}

# ---------- Persist state ----------
logger.info("Persisting scan state")
try:
    # Update last successful scan timestamp (use end_time for next incremental run)
    end_time_iso = final_stats["scan_window"]["end_time"] or run_end_time.isoformat()
    update_state("last_scan_run", end_time_iso)
    logger.info(f"Updated last_scan_run to {end_time_iso}")

    # Store run statistics for historical tracking
    state_key = f"scan_stats_{run_id}"
    update_state(state_key, final_stats)
    logger.info(f"Persisted run statistics to state key: {state_key}")

    # Update scan count
    scan_count = get_state("total_scan_runs", default=0)
    update_state("total_scan_runs", int(scan_count) + 1)
    logger.info(f"Total scan runs: {int(scan_count) + 1}")

except Exception as e:
    logger.error(f"Failed to persist state: {e}")
    logger.warning("Scan state not updated; next run may re-scan same window")

# ---------- Final report ----------
logger.info("=" * 80)
logger.info(f"DAILY PAPER SCANNER RUN SUMMARY: {run_id}")
logger.info("=" * 80)

logger.info(f"Run time: {final_stats['elapsed_minutes']:.2f} minutes")
logger.info(
    f"Scan window: {final_stats['scan_window']['window_hours']:.1f} hours "
    f"({final_stats['scan_window']['start_time']} to {final_stats['scan_window']['end_time']} UTC)"
)
logger.info("")

logger.info("FETCH STAGE:")
logger.info(f"  arXiv papers fetched: {final_stats['fetch']['arxiv_papers']}")
logger.info(f"  Semantic Scholar papers fetched: {final_stats['fetch']['semantic_scholar_papers']}")
logger.info(f"  Total fetched: {final_stats['fetch']['total_fetched']}")
logger.info("")

logger.info("BATCH DEDUPLICATION:")
logger.info(f"  Exact duplicates removed: {final_stats['batch_dedup']['exact_duplicates_removed']}")
logger.info(f"  Title-similar duplicates removed: {final_stats['batch_dedup']['title_duplicates_removed']}")
logger.info(f"  Total duplicates removed: {final_stats['batch_dedup']['total_duplicates_removed']}")
logger.info(f"  Unique papers after batch dedup: {final_stats['batch_dedup']['unique_papers']}")
logger.info(f"  Deduplication rate: {final_stats['batch_dedup']['deduplication_rate_pct']}%")
logger.info("")

logger.info("NOTION DUPLICATE CHECK:")
logger.info(f"  Papers checked against Notion DB: {final_stats['notion_check']['total_checked']}")
logger.info(f"  Duplicates found in Notion: {final_stats['notion_check']['duplicates_found']}")
logger.info(f"    By DOI: {final_stats['notion_check']['duplicates_by_doi']}")
logger.info(f"    By arXiv ID: {final_stats['notion_check']['duplicates_by_arxiv_id']}")
logger.info(f"    By title similarity: {final_stats['notion_check']['duplicates_by_title']}")
logger.info(f"  New papers (not in Notion): {final_stats['notion_check']['new_papers']}")
logger.info(f"  Check errors: {final_stats['notion_check']['check_errors']}")
logger.info("")

logger.info("SUMMARY GENERATION:")
logger.info(f"  Papers summarized: {final_stats['summary']['total_papers']}")
logger.info(f"  LLM summaries: {final_stats['summary']['llm_summaries']}")
logger.info(f"  Fallback summaries: {final_stats['summary']['fallback_summaries']}")
if final_stats["summary"]["llm_tokens_used"] > 0:
    logger.info(f"  LLM tokens used: {final_stats['summary']['llm_tokens_used']:,}")
logger.info(f"  Summary errors: {final_stats['summary']['summary_errors']}")
logger.info("")

logger.info("NOTION WRITE STAGE:")
if dry_run:
    logger.info(f"  DRY RUN MODE: {final_stats['write']['dry_run_skipped']} writes simulated")
else:
    logger.info(f"  Papers to write: {final_stats['write']['total_papers']}")
    logger.info(f"  Successful writes: {final_stats['write']['successful_writes']}")
    logger.info(f"  Failed writes: {final_stats['write']['failed_writes']}")
    if final_stats["write"]["total_papers"] > 0:
        sr = final_stats["write"]["successful_writes"] / final_stats["write"]["total_papers"] * 100
        logger.info(f"  Write success rate: {sr:.1f}%")
logger.info("")

logger.info("PIPELINE HEALTH:")
logger.info(f"  Total errors (all stages): {final_stats['pipeline']['total_errors']}")
logger.info(f"  Pipeline status: {'SUCCESS' if final_stats['pipeline']['success'] else 'PARTIAL FAILURE'}")
logger.info("")

if dry_run:
    logger.info(f"✓ DRY RUN COMPLETE: {run_id}")
    logger.info(f"  Would have created {final_stats['write']['dry_run_skipped']} new paper records in Notion.")
elif final_stats["pipeline"]["success"]:
    logger.info(f"✓ SCAN COMPLETE: {run_id}")
    logger.info(f"  Created {final_stats['write']['successful_writes']} new paper records in Notion.")
else:
    logger.warning(f"⚠ SCAN COMPLETE WITH ERRORS: {run_id}")
    logger.warning(f"  Created {final_stats['write']['successful_writes']} records; {final_stats['write']['failed_writes']} writes failed.")

logger.info("=" * 80)

# Store for programmatic access
scan_summary = final_stats

logger.info("Cell 12 complete: state persisted, statistics reported")
logger.info(f"Daily paper scanner run {run_id} finished.")


2026-01-25 14:17:21 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 | Starting Cell 12: Persist state and report statistics
2026-01-25 14:17:21 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 | Scan run 2ac5680b-7a63-488f-a211-1860c73ee2f8 completed in 26.04 minutes
2026-01-25 14:17:21 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 | Persisting scan state
2026-01-25 14:17:21 | ERROR    | 2ac5680b-7a63-488f-a211-1860c73ee2f8 | Failed to persist state: name 'update_state' is not defined
2026-01-25 14:17:21 | WARNING  | 2ac5680b-7a63-488f-a211-1860c73ee2f8 | Scan state not updated; next run may re-scan same window
2026-01-25 14:17:21 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 | ================================================================================
2026-01-25 14:17:21 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 | DAILY PAPER SCANNER RUN SUMMARY: 2ac5680b-7a63-488f-a211-1860c73ee2f8
2026-01-25 14:17:21 | INFO     | 2ac5680b-7a63-488f-a211-1860c73ee2f8 | =======